# ChronoQuality-CRAC V7 + Reviewer-Requested Validation — Compact Full Kaggle Notebook

**Source:** original 112-cell V7 experiment code retained in execution order; supplemented with the standalone reviewer analyses. Compactness is achieved by merging notebook cells and reducing navigation, **not removing original V7 research stages**.

**Inputs:** five historical `cms_hospital_patient_satisfaction_YYYY.csv` files (2016–2020) attached in Kaggle. Uses CPU (no GPU). **Outputs:** original `ChronoQuality_RUN_007_Results.zip`, reviewer validation results, and combined archive. It must be run from top to bottom. The original five-CSV results are not pre-populated. Experiments must complete and be checked before using any numbers in a revised paper.

**Note:** V7 analyses and reviewer diagnostics are deliberately preserved as separate computational branches. Interpret adjusted hospital-type estimates as observational and sensitive to specification; certificates only apply to enumerated scoring policies. Historical HCAHPS construct validation and practical analyst evaluation cannot be supplied by code alone.

## V7 original — A. Input audit, provenance, and leakage-safe feature semantics / B. Real temporal comparability and score construction

*All original code for this stage is preserved.*

In [ ]:
# Original V7 cell 2 (merged stage)
import os, glob, json, time, warnings, platform, sys, hashlib, zipfile, math
from pathlib import Path
import numpy as np, pandas as pd
from scipy.stats import kendalltau, spearmanr
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
SEED=42
rng=np.random.default_rng(SEED)
DATA=Path('/kaggle/input/datasets/abrambeyer/us-hospital-customer-satisfaction-20162020')
if not DATA.exists():
    options=list(Path('/kaggle/input').rglob('cms_hospital_patient_satisfaction_2016.csv'))
    assert options, 'Attach the abrambeyer Kaggle dataset before running.'
    DATA=options[0].parent
ROOT=Path('/kaggle/working/chronoquality_runs/RUN_004')
for sub in ['qc','data','metrics','tables','figures','logs','predictions','comparisons','bootstrap','perturbations','ablations','configs']:
    (ROOT/sub).mkdir(parents=True,exist_ok=True)
CONFIG={'seed':SEED,'development_years':[2016,2017,2018],'test_years':[2019,2020],
        'min_shared_measures':3,'direction_margin_pp':2.0,'min_fraction_agree':0.60,
        'max_pair_hospitals':350,'perturbation_repeats':15,
        'perturbation_rates':[0.1,0.2,0.4,0.6],'bootstrap_reps':2000,
        'pilot_bootstrap_reps':100,'max_plot_hospitals':35,'version':'4.0-ranking-audit'}
(ROOT/'configs'/'config.json').write_text(json.dumps(CONFIG,indent=2))
print('DATA=',DATA,'OUTPUT=',ROOT,'CPU=',os.cpu_count())

# Original V7 cell 4 (merged stage)
FILES=sorted(DATA.glob('cms_hospital_patient_satisfaction_*.csv'))
assert len(FILES)==5, f'Expected five release CSVs, got {len(FILES)}'
COLUMNS=['Facility ID','Facility Name','State','Hospital Type','HCAHPS Measure ID','HCAHPS Question',
         'HCAHPS Answer Description','HCAHPS Answer Percent','Number of Completed Surveys',
         'Survey Response Rate Percent','Start Date','End Date','Year','Hospital overall rating','Hospital Ownership']
frames=[]; manifest=[]
for f in FILES:
    d=pd.read_csv(f,usecols=lambda c:c in COLUMNS,low_memory=False,dtype={'Facility ID':'string','HCAHPS Measure ID':'string'})
    assert (set(COLUMNS)-{'Hospital Ownership'}).issubset(d.columns),f'Missing columns in {f.name}: {set(COLUMNS)-set(d.columns)}'
    fileyear=int(f.stem.split('_')[-1]); assert set(d.Year.dropna().unique())=={fileyear}
    d['source_file']=f.name
    frames.append(d)
    manifest.append({'file':f.name,'sha256':hashlib.sha256(f.read_bytes()).hexdigest(),
                     'rows':len(d),'release_year':fileyear})
raw=pd.concat(frames,ignore_index=True)
assert len(raw)==1653683, f'Unexpected row count {len(raw)}; validate dataset version.'
(ROOT/'qc'/'input_manifest.json').write_text(json.dumps(manifest,indent=2))
raw['Start Date']=pd.to_datetime(raw['Start Date'],format='%m/%d/%Y',errors='coerce')
raw['End Date']=pd.to_datetime(raw['End Date'],format='%m/%d/%Y',errors='coerce')
raw['answer_num']=pd.to_numeric(raw['HCAHPS Answer Percent'],errors='coerce')
raw['survey_n']=pd.to_numeric(raw['Number of Completed Surveys'],errors='coerce')
raw['response_pct']=pd.to_numeric(raw['Survey Response Rate Percent'],errors='coerce')
raw['measure']=raw['HCAHPS Measure ID'].str.strip()
raw['facility']=raw['Facility ID'].str.zfill(6)
raw['question']=raw['HCAHPS Question'].astype('string').str.strip().str.lower().str.replace(r'\s+',' ',regex=True)
raw['answer_desc']=raw['HCAHPS Answer Description'].astype('string').str.strip().str.lower().str.replace(r'\s+',' ',regex=True)
raw['is_positive']=raw['measure'].str.endswith('_A_P',na=False)
window_audit=(raw.groupby('Year').agg(rows=('facility','size'),hospitals=('facility','nunique'),
    measures=('measure','nunique'),start_unique=('Start Date','nunique'),end_unique=('End Date','nunique'),
    first_start=('Start Date','min'),last_end=('End Date','max'),numeric_positive_rows=('answer_num','count')).reset_index())
window_audit.to_csv(ROOT/'qc'/'release_audit.csv',index=False)
print(window_audit.to_string(index=False));print('positive rows',raw.is_positive.sum())
assert (window_audit['start_unique']==1).all() and (window_audit['end_unique']==1).all(), 'Window assumption changed: revise study design.'

# Original V7 cell 5 (merged stage)
# Semantic catalog: compare BOTH question and answer-description normalized text.
pos=raw.loc[raw.is_positive].copy()
pos['sem_key']=pos['question'].fillna('')+' || '+pos['answer_desc'].fillna('')
sem_counts=pos.groupby('measure').agg(n_sem=('sem_key','nunique'),n_years=('Year','nunique'),rows=('facility','size')).reset_index()
sem_counts.to_csv(ROOT/'qc'/'semantic_variation_by_measure.csv',index=False)
sem_rows=(pos[['Year','measure','sem_key']].drop_duplicates()
          .sort_values(['measure','Year']))
sem_rows.to_csv(ROOT/'qc'/'measure_definitions_by_release.csv',index=False)
# Year-specific key includes description: never merge a changed definition into same valid concept.
pos['concept']=pos['measure']+' || '+pos['sem_key']
# Stable development vocabulary is selected ONLY from 2016-18 and locked for held-out years.
dev=pos[pos.Year.isin(CONFIG['development_years'])]
dev_counts=dev.groupby('concept')['Year'].nunique()
LOCKED_CONCEPTS=set(dev_counts[dev_counts==3].index)
if not LOCKED_CONCEPTS:
    # fallback defined without test-year peeking, requires at least 2 development years
    LOCKED_CONCEPTS=set(dev_counts[dev_counts>=2].index)
assert LOCKED_CONCEPTS,'No stable positive concepts found in development years.'
(ROOT/'configs'/'locked_concepts.json').write_text(json.dumps(sorted(LOCKED_CONCEPTS),indent=2))
print('Stable development concepts:',len(LOCKED_CONCEPTS),'Number of codes whose text changed:',sum(sem_counts.n_sem>1))
valid=pos[pos['concept'].isin(LOCKED_CONCEPTS) & pos['answer_num'].between(0,100) & pos.facility.notna()].copy()
valid=valid.drop_duplicates(['Year','facility','concept'],keep=False)  # ambiguous duplicated values discarded, not averaged
panel=valid.pivot(index=['Year','facility'],columns='concept',values='answer_num').sort_index()
assert panel.index.is_unique
panel.to_pickle(ROOT/'data'/'panel.pkl')
meta=valid.groupby(['Year','facility'],sort=False).agg(state=('State','first'),hospital_type=('Hospital Type','first'),
 survey_n=('survey_n','first'),response_pct=('response_pct','first'),start=('Start Date','first'),end=('End Date','first')).reindex(panel.index)
meta.to_csv(ROOT/'data'/'metadata.csv')
coverage=panel.notna().sum(axis=1).rename('available_measures').reset_index()
coverage.to_csv(ROOT/'qc'/'availability_per_hospital_year.csv',index=False)
print('Panel:',panel.shape,'valid numeric coverage:',__import__('builtins').round(panel.notna().mean().mean(),3))

# Original V7 cell 7 (merged stage)
windows=raw.groupby('Year')[['Start Date','End Date']].first().sort_index()
def intersection_days(a,b,c,d):
    return max(0,(min(b,d)-max(a,c)).days+1)
pairs=[]
for y1 in windows.index:
 for y2 in windows.index:
  if y1>=y2: continue
  a,b=windows.loc[y1,['Start Date','End Date']]; c,d=windows.loc[y2,['Start Date','End Date']]
  i=intersection_days(a,b,c,d)
  l1=(b-a).days+1;l2=(d-c).days+1
  pairs.append({'release_1':int(y1),'release_2':int(y2),'overlap_days':i,
                'overlap_coefficient':i/min(l1,l2),'jaccard_days':i/(l1+l2-i)})
overlap=pd.DataFrame(pairs); overlap.to_csv(ROOT/'tables'/'temporal_overlap.csv',index=False)
print(overlap.to_string(index=False))
# Baseline hospital-year score tables, enforcing minimum available items
scores=pd.DataFrame(index=panel.index)
scores['simple_mean']=panel.mean(axis=1)
scores['median']=panel.median(axis=1)
scores['measure_coverage']=panel.notna().sum(axis=1)
scores['all_dev_locked_mean']=panel.reindex(columns=sorted(LOCKED_CONCEPTS)).mean(axis=1)
scores.loc[scores.measure_coverage<CONFIG['min_shared_measures'],['simple_mean','median','all_dev_locked_mean']]=np.nan
scores=scores.join(meta)
scores.to_csv(ROOT/'tables'/'hospital_year_scores.csv')
print('Hospital-years:',len(scores))

## V7 original — C. Comparability-constrained partial order / D. Baselines and independent, controlled benchmark

*All original code for this stage is preserved.*

In [ ]:
# Original V7 cell 9 (merged stage)
def pairwise_method(X, margin=2.0, min_shared=3, min_agree=.60, use_agreement=True):
    # X: N x M, compare all unordered pairs without n*n*m memory
    A=np.asarray(X,dtype='float32'); n,m=A.shape
    src=[];dst=[];diffs=[];counts=[];agrees=[];eligible=0
    for i in range(n-1):
        a=A[i]; b=A[i+1:]; common=np.isfinite(b)&np.isfinite(a)
        cnt=common.sum(axis=1)
        diff=np.where(common,a-b,0)
        avg=diff.sum(axis=1)/np.maximum(1,cnt)
        agreement=np.maximum((diff>0).sum(axis=1),(diff<0).sum(axis=1))/np.maximum(1,cnt)
        good=(cnt>=min_shared)
        eligible+=int(good.sum())
        accepted=good & (abs(avg)>=margin) & ((agreement>=min_agree) if use_agreement else True)
        js=np.flatnonzero(accepted)+i+1
        src.extend(np.where(avg[accepted]>0,i,js).tolist())
        dst.extend(np.where(avg[accepted]>0,js,i).tolist())
        diffs.extend(np.abs(avg[accepted]).tolist());counts.extend(cnt[accepted].tolist());agrees.extend(agreement[accepted].tolist())
    edges=pd.DataFrame({'better_idx':src,'worse_idx':dst,'gap_pp':diffs,'shared_items':counts,'fraction_agree':agrees})
    total=n*(n-1)//2
    return edges,{'hospitals':n,'total_pairs':total,'eligible_shared_pairs':eligible,'supported_pairs':len(edges),
                  'supported_fraction':len(edges)/total if total else np.nan}
def select_hospitals(year,cap=350):
    sub=panel.loc[year]
    ids=np.array(sorted(sub.index))
    rr=np.random.default_rng(SEED+int(year))
    ids=np.sort(rr.choice(ids,size=min(len(ids),cap),replace=False))
    return sub.loc[ids]
edge_summaries=[]
for year in sorted(panel.index.get_level_values(0).unique()):
    sub=select_hospitals(year,CONFIG['max_pair_hospitals'])
    e,s=pairwise_method(sub.values,margin=CONFIG['direction_margin_pp'],min_shared=CONFIG['min_shared_measures'],min_agree=CONFIG['min_fraction_agree'])
    e['better_hospital']=np.asarray(sub.index)[e.better_idx.values] if len(e) else []
    e['worse_hospital']=np.asarray(sub.index)[e.worse_idx.values] if len(e) else []
    e.to_csv(ROOT/'comparisons'/f'edges_{year}.csv',index=False)
    s['year']=int(year);edge_summaries.append(s)
pair_summary=pd.DataFrame(edge_summaries);pair_summary.to_csv(ROOT/'metrics'/'comparison_coverage.csv',index=False)
print(pair_summary.to_string(index=False))

# Original V7 cell 11 (merged stage)
def edge_keys(edges,ids):
    return set((str(ids[int(a)]),str(ids[int(b)])) for a,b in zip(edges.better_idx,edges.worse_idx))
def compare_edges(base,changed):
    # evaluate agreement only on decisions present in original reference
    b=set(base);c=set(changed);rev={(v,u) for u,v in b}
    return {'reference_edges':len(b),'candidate_edges':len(c),'retention':len(b&c)/len(b) if b else np.nan,
            'reversal_rate':len(rev&c)/len(b) if b else np.nan,
            'new_decision_fraction':len(c-b)/len(c) if c else np.nan}
def simple_edges(X,mode='mean',margin=2.0,min_shared=3):
    x=np.asarray(X,float); n=len(x);sc=np.nanmedian(x,axis=1) if mode=='median' else np.nanmean(x,axis=1)
    src=[];dst=[]
    for i in range(n-1):
        dif=sc[i]-sc[i+1:]
        cnt=np.sum(np.isfinite(x[i])&np.isfinite(x[i+1:]),axis=1)
        valid=np.isfinite(dif)&(abs(dif)>=margin)
        if mode=='matched_mean': valid &= cnt>=min_shared
        js=np.flatnonzero(valid)+i+1
        src.extend(np.where(dif[valid]>0,i,js).tolist());dst.extend(np.where(dif[valid]>0,js,i).tolist())
    return pd.DataFrame({'better_idx':src,'worse_idx':dst})
# Compare baseline disagreement with full constrained method, NOT with clinical ground truth.
baseline_rows=[]
for year in [2016,2017,2018,2019,2020]:
    sub=select_hospitals(year,CONFIG['max_pair_hospitals']);ids=sub.index.to_numpy()
    full,_=pairwise_method(sub.to_numpy())
    ref=edge_keys(full,ids)
    alternatives={'mean':simple_edges(sub,'mean'), 'median':simple_edges(sub,'median'),
       'matched_mean':simple_edges(sub,'matched_mean'),
       'overlap_only':pairwise_method(sub.to_numpy(),margin=2,min_shared=3,min_agree=0,use_agreement=False)[0],
       'majority_only':pairwise_method(sub.to_numpy(),margin=0,min_shared=3,min_agree=.60)[0]}
    for name,ee in alternatives.items():
        d=compare_edges(ref,edge_keys(ee,ids));d.update({'year':year,'baseline':name});baseline_rows.append(d)
baselines=pd.DataFrame(baseline_rows);baselines.to_csv(ROOT/'tables'/'baseline_agreement.csv',index=False)
print(baselines.head(10).to_string(index=False))

# Original V7 cell 13 (merged stage)
conditions=[];nrep=CONFIG['perturbation_repeats']
variants={
 'FULL':dict(margin=2.0,min_shared=3,min_agree=.60,use_agreement=True),
 'NO_AGREEMENT':dict(margin=2.0,min_shared=3,min_agree=0.0,use_agreement=False),
 'NO_MARGIN':dict(margin=0.0,min_shared=3,min_agree=.60,use_agreement=True),
 'MIN_ONE':dict(margin=2.0,min_shared=1,min_agree=.60,use_agreement=True),
 'STRICT_75':dict(margin=2.0,min_shared=3,min_agree=.75,use_agreement=True),
}
for year in [2016,2017,2018,2019,2020]:
    sub=select_hospitals(year,250); X=sub.to_numpy(dtype=float); ids=sub.index.to_numpy()
    full,_=pairwise_method(X,**variants['FULL']);baseline=edge_keys(full,ids)
    for rate in CONFIG['perturbation_rates']:
        for rep in range(nrep):
            rr=np.random.default_rng(SEED+year*100000+int(rate*1000)*100+rep)
            mask=rr.random(X.shape)<rate;damaged=X.copy();damaged[mask]=np.nan
            for name,kwargs in variants.items():
                e,_=pairwise_method(damaged,**kwargs)
                d=compare_edges(baseline,edge_keys(e,ids))
                d.update({'year':year,'rate':rate,'repeat':rep,'variant':name,'simulation':'random_measure_loss'})
                conditions.append(d)
pert=pd.DataFrame(conditions);pert.to_csv(ROOT/'perturbations'/'measure_loss_all.csv',index=False)
abl=pert.groupby(['year','rate','variant'],as_index=False).agg(retention_mean=('retention','mean'),reversal_mean=('reversal_rate','mean'),new_decision_mean=('new_decision_fraction','mean'))
abl.to_csv(ROOT/'ablations'/'ablation_aggregates.csv',index=False)
print(abl.query("variant == 'FULL'").to_string(index=False))

## V7 original — F. Temporal generalization, longitudinal stability, and real reporting windows / G. Controlled temporal metadata mismatch

*All original code for this stage is preserved.*

In [ ]:
# Original V7 cell 15 (merged stage)
long_rows=[]
for a,b in [(2016,2017),(2017,2018),(2018,2019),(2019,2020),(2016,2020)]:
    A=scores.loc[a,['simple_mean','measure_coverage']].rename(columns={'simple_mean':'score_a','measure_coverage':'m_a'})
    B=scores.loc[b,['simple_mean','measure_coverage']].rename(columns={'simple_mean':'score_b','measure_coverage':'m_b'})
    j=A.join(B,how='inner').dropna(subset=['score_a','score_b'])
    rho=spearmanr(j.score_a,j.score_b).statistic if len(j)>3 else np.nan
    tau=kendalltau(j.score_a,j.score_b).statistic if len(j)>3 else np.nan
    topa=set(j.nlargest(min(100,len(j)),'score_a').index);topb=set(j.nlargest(min(100,len(j)),'score_b').index)
    row={'year_a':a,'year_b':b,'matched_hospitals':len(j),'spearman':rho,'kendall_tau':tau,
        'top100_jaccard':len(topa&topb)/len(topa|topb) if topa|topb else np.nan,
        'mean_abs_score_change':(j.score_a-j.score_b).abs().mean()}
    row.update(overlap.set_index(['release_1','release_2']).loc[(a,b),['overlap_days','overlap_coefficient']].to_dict())
    long_rows.append(row)
longitudinal=pd.DataFrame(long_rows);longitudinal.to_csv(ROOT/'tables'/'longitudinal_stability.csv',index=False)
# Held-out results distinguished in output; no model training is involved.
longitudinal['evaluation_role']=np.where(longitudinal.year_b<=2018,'development_descriptive','held_out_descriptive')
longitudinal.to_csv(ROOT/'metrics'/'temporal_holdout.csv',index=False)
print(longitudinal.to_string(index=False))

# Original V7 cell 17 (merged stage)
time_rows=[]
for year in [2016,2017,2018,2019,2020]:
    ids=np.asarray(sorted(meta.loc[year].index)); N=len(ids)
    orig_start=meta.loc[year,'start'].reindex(ids).to_numpy(dtype='datetime64[ns]')
    orig_end=meta.loc[year,'end'].reindex(ids).to_numpy(dtype='datetime64[ns]')
    for rate in [0,.1,.2,.4,.6]:
      for rep in range(20):
        rr=np.random.default_rng(SEED+year*100000+int(rate*100)*100+rep)
        injected=rr.random(N)<rate
        new_start=orig_start.copy();new_end=orig_end.copy()
        # shifts of 30/90/180 days; metadata only, never modifying response values
        offsets=rr.choice([30,90,180],size=N)
        shift=offsets.astype('timedelta64[D]')
        new_start[injected]+=shift[injected];new_end[injected]+=shift[injected]
        flagged=(new_start!=orig_start)|(new_end!=orig_end)
        tp=int(np.sum(flagged&injected)); fp=int(np.sum(flagged&~injected));fn=int(np.sum(~flagged&injected))
        time_rows.append({'year':year,'rate':rate,'repeat':rep,'tp':tp,'fp':fp,'fn':fn,
                          'recall':tp/(tp+fn) if tp+fn else np.nan,'precision':tp/(tp+fp) if tp+fp else np.nan})
time_eval=pd.DataFrame(time_rows);time_eval.to_csv(ROOT/'perturbations'/'temporal_metadata_injection.csv',index=False)
print(time_eval.groupby('rate')[['recall','precision']].mean())

# Original V7 cell 19 (merged stage)
boot_rows=[]
N_BOOT=CONFIG['bootstrap_reps']
for a,b in [(2016,2017),(2017,2018),(2018,2019),(2019,2020)]:
    A=scores.loc[a,'simple_mean'];B=scores.loc[b,'simple_mean'];matched=pd.concat([A.rename('a'),B.rename('b')],axis=1).dropna()
    vals=(matched.b-matched.a).to_numpy(float)
    if len(vals)<10:continue
    rr=np.random.default_rng(SEED+a*10+b)
    # batched bootstrap avoids huge allocations and guarantees 2000 replicates
    estimates=[]
    for k in range(0,N_BOOT,100):
        ix=rr.integers(0,len(vals),size=(min(100,N_BOOT-k),len(vals)))
        estimates.extend(vals[ix].mean(axis=1).tolist())
    boot_rows.append({'year_a':a,'year_b':b,'n_matched_hospitals':len(vals),
       'mean_change':vals.mean(),'ci_low':np.quantile(estimates,.025),'ci_high':np.quantile(estimates,.975),
       'bootstrap_reps':N_BOOT})
boot=pd.DataFrame(boot_rows);boot.to_csv(ROOT/'bootstrap'/'hospital_cluster_bootstrap.csv',index=False)
print(boot.to_string(index=False))

## V7 original — I. Perturbation significance, subgroup and methodological diagnostics / J. Runtime and scalability tests

*All original code for this stage is preserved.*

In [ ]:
# Original V7 cell 21 (merged stage)
stat_rows=[]
for (year,rate),g in pert.groupby(['year','rate']):
    g=g.pivot(index='repeat',columns='variant',values='retention')
    if {'FULL','NO_AGREEMENT'}.issubset(g):
        d=(g['FULL']-g['NO_AGREEMENT']).dropna().to_numpy()
        stat_rows.append({'year':year,'rate':rate,'n_paired':len(d),'mean_retention_delta':d.mean() if len(d) else np.nan,
                          'delta_ci_low':np.quantile(d,.025) if len(d) else np.nan,'delta_ci_high':np.quantile(d,.975) if len(d) else np.nan})
stat=pd.DataFrame(stat_rows);stat.to_csv(ROOT/'metrics'/'paired_ablation_deltas.csv',index=False)
subgroup=[]
for (year,kind),z in scores.reset_index().groupby(['Year','hospital_type'],dropna=False):
    if len(z)<20:continue
    subgroup.append({'year':year,'hospital_type':str(kind),'n_hospitals':len(z),
                     'mean_score':z.simple_mean.mean(),'median_score':z.simple_mean.median(),
                     'median_coverage':z.measure_coverage.median()})
pd.DataFrame(subgroup).to_csv(ROOT/'tables'/'hospital_type_descriptive.csv',index=False)
# Data model QC: repeated hospital-wide metadata duplicated across measure rows is not independent observations
qc={'total_rows':int(len(raw)),'hospital_years':int(len(panel)),'locked_concepts':len(LOCKED_CONCEPTS),
    'positive_numeric_rows':int(valid.shape[0]),'missing_positive_pct':float(100*(1-panel.notna().mean().mean())),
    'within_release_window_heterogeneity':bool((window_audit.start_unique>1).any() or (window_audit.end_unique>1).any()),
    'used_overall_hospital_rating_in_algorithm':False,'test_years_used_for_concept_selection':False,
    'claim_independent_clinical_ground_truth':False}
(ROOT/'qc'/'methodology_assertions.json').write_text(json.dumps(qc,indent=2))
assert not qc['within_release_window_heterogeneity'], 'Revise the hypothesis and methods if window heterogeneity is present.'
print(json.dumps(qc,indent=2))

# Original V7 cell 23 (merged stage)
benchmark=[]
sub=select_hospitals(2020,CONFIG['max_pair_hospitals'])
for size in [50,100,200,500,1000,2000,4000]:
    if size>len(sub):continue
    st=time.perf_counter();e,s=pairwise_method(sub.iloc[:size].to_numpy());sec=time.perf_counter()-st
    benchmark.append({'hospital_count':size,'pair_count':size*(size-1)//2,'runtime_seconds':sec,'supported_pairs':len(e)})
scalability=pd.DataFrame(benchmark);scalability.to_csv(ROOT/'metrics'/'scalability.csv',index=False)
print(scalability.to_string(index=False))

# Original V7 cell 25 (merged stage)
plt.rcParams.update({'font.size':10,'figure.dpi':130})
def savefigure(name):
    plt.tight_layout();plt.savefig(ROOT/'figures'/name,dpi=300,bbox_inches='tight');plt.close()
fig,ax=plt.subplots(figsize=(8,4));ax.bar(window_audit.Year.astype(str),window_audit.measures)
ax.set(xlabel='Release year',ylabel='Unique HCAHPS measure IDs',title='Indicator availability across annual releases');savefigure('F1_measure_catalog.png')
fig,ax=plt.subplots(figsize=(8,4));ax.bar(window_audit.Year.astype(str),window_audit.hospitals)
ax.set(xlabel='Release year',ylabel='Hospitals',title='Hospital coverage by release');savefigure('F2_hospital_coverage.png')
fig,ax=plt.subplots(figsize=(8,4));
for name,z in pert.query('year == 2020').groupby('variant'):
    agg=z.groupby('rate').retention.mean();ax.plot(agg.index,agg.values,marker='o',label=name)
ax.set(xlabel='Artificial item deletion fraction',ylabel='Reference decisions retained',title='Simulated measure-loss robustness — 2020');ax.legend(fontsize=7);savefigure('F3_perturbation_retention.png')
fig,ax=plt.subplots(figsize=(8,4));ax.plot(longitudinal.year_b, longitudinal.spearman,marker='o')
ax.set(xlabel='Second release in pair',ylabel='Matched-hospital Spearman correlation',title='Descriptive longitudinal score stability');savefigure('F4_year_stability.png')
fig,ax=plt.subplots(figsize=(8,4));ax.plot(pair_summary.year,pair_summary.supported_fraction,marker='o',label='Supported')
ax.plot(pair_summary.year,pair_summary.eligible_shared_pairs/pair_summary.total_pairs,marker='o',label='Adequately overlapping')
ax.set(xlabel='Release year',ylabel='Fraction of evaluated pairs',title='Comparability coverage');ax.legend();savefigure('F5_pairwise_coverage.png')
fig,ax=plt.subplots(figsize=(8,4));ax.plot(scalability.hospital_count,scalability.runtime_seconds,marker='o')
ax.set(xlabel='Number of hospitals',ylabel='Seconds',title='Measured CPU pairwise runtime');savefigure('F6_scalability.png')
summary={'source_file_count':5,'release_years':[2016,2017,2018,2019,2020],
         'n_source_rows':len(raw),'n_hospital_years':len(panel),'n_locked_concepts':len(LOCKED_CONCEPTS),
         'n_bootstrap_reps':N_BOOT,'n_measure_loss_replicates_per_condition':nrep,
         'limitations':['Five standardized within-release measurement windows; no observed within-release mismatch',
                        'HCAHPS positive percentages are aggregate, not individual response records',
                        'No independent hospital-quality ground truth, clinical outcomes, or deployment impact',
                        'Controlled perturbations are simulations, not observed causal changes',
                        'Restricted scope to patient-experience indicators and stable development definitions']}
(ROOT/'metrics'/'run_summary.json').write_text(json.dumps(summary,indent=2))
print('Created',len(list((ROOT/'figures').glob('*.png'))),'figures')

# Original V7 cell 27 (merged stage)
required=[ROOT/'qc'/'input_manifest.json',ROOT/'configs'/'locked_concepts.json',ROOT/'tables'/'longitudinal_stability.csv',
          ROOT/'metrics'/'comparison_coverage.csv',ROOT/'metrics'/'temporal_holdout.csv',ROOT/'ablations'/'ablation_aggregates.csv',
          ROOT/'perturbations'/'temporal_metadata_injection.csv',ROOT/'bootstrap'/'hospital_cluster_bootstrap.csv',
          ROOT/'metrics'/'scalability.csv',ROOT/'metrics'/'run_summary.json']
assert all(x.exists() and x.stat().st_size>0 for x in required),'Missing required artifacts'
notes="""# ChronoQuality result interpretation

- No diagnostic or clinical ground truth labels exist in this collection.
- Graph edges mean prespecified consistent favorable-response evidence, not significance.
- Measure-removal and timestamp interventions are synthetic.
- All annual within-release windows are synchronized; do not claim observed window incompatibility within a year.
- Compare 2019–2020 using locked 2016–2018 semantic vocabulary.
- Multiple HCAHPS rows per hospital are repeated measurements, not independent hospitals.
- Survey counts are not exact per-item response denominators.
- Read empirical output before claiming improvement or journal readiness.
"""
(ROOT/'RESULT_INTERPRETATION.md').write_text(notes)
archive=Path('/kaggle/working/ChronoQuality_RUN_001_Results.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as z:
 for file in sorted(ROOT.rglob('*')):
    if file.is_file():z.write(file,arcname=str(file.relative_to(ROOT.parent)))
print('COMPLETE:',archive,'size(MB)',__import__('builtins').round(archive.stat().st_size/1024**2,2))
print('Artifacts:',sum(1 for p in ROOT.rglob('*') if p.is_file()))

## V7 original — M. V2 — Independent controlled comparison benchmark (locked simulator) / N. V2 — Matched-coverage and paired uncertainty evaluation

*All original code for this stage is preserved.*

In [ ]:
# Original V7 cell 29 (merged stage)
# This simulation is an independently specified data-generating mechanism, not patient ground truth.
# Distinct random seeds for development and evaluation. Model parameters remain frozen.
import scipy.stats as ss
from itertools import product
SIM_CONFIG={'n_hospitals':650,'n_items':6,'seeds_dev':[111,112,113],
 'seeds_test':[9211,9212,9213,9214,9215],
 'missing_rates':[0.0,0.15,0.35,0.55],
 'semantic_corruption_rates':[0.0,0.1,0.3],
 'true_margin':3.0,'evaluated_pairs_per_scenario':35000}
(ROOT/'configs'/'simulation_protocol.json').write_text(json.dumps(SIM_CONFIG,indent=2))
def generate_locked_scenario(seed,missing_rate,semantic_corruption):
    rg=np.random.default_rng(seed); nh=SIM_CONFIG['n_hospitals']; nm=SIM_CONFIG['n_items']
    theta=rg.uniform(35,85,nh)
    item_offset=rg.normal(0,5,nm)
    # heterogeneous, but partly informative item response, errors independent of scoring algorithm
    observed=theta[:,None]+item_offset+rg.normal(0,6,(nh,nm))
    observed=np.clip(observed,0,100)
    bad=rg.random((nh,nm))<semantic_corruption
    # corrupted concepts are truly non-equivalent: opposite scoring direction for some records
    observed=np.where(bad,100-observed,observed)
    missing=rg.random((nh,nm)) < (missing_rate + 0.15*(theta[:,None]<48))
    observed[missing]=np.nan
    # independently known indicator definitions: corrupted entries need invalidation
    validated=observed.copy();validated[bad]=np.nan
    return theta,observed,validated,bad

def vector_pairs(X,pair_i,pair_j,method='full',margin=2.0,min_shared=3,min_agree=.60):
    A=np.asarray(X,dtype=float)
    L=A[pair_i];R=A[pair_j];mask=np.isfinite(L)&np.isfinite(R)
    cnt=mask.sum(axis=1)
    dif=np.where(mask,L-R,0)
    matched=dif.sum(axis=1)/np.maximum(cnt,1)
    agree=np.maximum((dif>0).sum(axis=1),(dif<0).sum(axis=1))/np.maximum(cnt,1)
    avg=np.nanmean(A,axis=1)
    med=np.nanmedian(A,axis=1)
    if method=='unmatched_mean':
        d=avg[pair_i]-avg[pair_j];valid=np.isfinite(d)&(np.abs(d)>=margin)
    elif method=='unmatched_median':
        d=med[pair_i]-med[pair_j];valid=np.isfinite(d)&(np.abs(d)>=margin)
    elif method=='matched_mean':
        d=matched;valid=(cnt>=min_shared)&(np.abs(d)>=margin)
    elif method=='full':
        d=matched;valid=(cnt>=min_shared)&(np.abs(d)>=margin)&(agree>=min_agree)
    elif method=='stricter_majority':
        d=matched;valid=(cnt>=min_shared)&(np.abs(d)>=margin)&(agree>=.75)
    elif method=='majority_only':
        d=matched;valid=(cnt>=min_shared)&(agree>=min_agree)&(np.abs(d)>0)
    else: raise ValueError(method)
    return np.where(valid,np.sign(d),0).astype('int8')

def summarize_predictions(pred,truth):
    supported=pred!=0
    valid=truth!=0
    keep=supported&valid
    return dict(coverage=float(supported.mean()),
       correctness=float((pred[keep]==truth[keep]).mean()) if keep.any() else np.nan,
       error=float((pred[keep]!=truth[keep]).mean()) if keep.any() else np.nan,
       false_direction_rate=float(((pred!=truth)&supported&valid).sum()/max(1,valid.sum())),
       pairs=int(len(pred)),supported=int(supported.sum()),eligible=int(valid.sum()))

SIM_METHODS=['unmatched_mean','unmatched_median','matched_mean','majority_only','stricter_majority','full']
records=[]
for phase,seeds in [('development',SIM_CONFIG['seeds_dev']),('locked_test',SIM_CONFIG['seeds_test'])]:
 for seed,loss,corrupt in product(seeds,SIM_CONFIG['missing_rates'],SIM_CONFIG['semantic_corruption_rates']):
    theta,X,V,bad=generate_locked_scenario(seed,loss,corrupt)
    rg=np.random.default_rng(seed+70000)
    i=rg.integers(0,len(theta),SIM_CONFIG['evaluated_pairs_per_scenario'])
    j=rg.integers(0,len(theta)-1,len(i));j+=(j>=i)
    delta=theta[i]-theta[j]
    truth=np.where(abs(delta)>=SIM_CONFIG['true_margin'],np.sign(delta),0).astype('int8')
    for variant,array in [('unverified_input',X),('verified_semantics',V)]:
     for method in SIM_METHODS:
      import warnings
      with warnings.catch_warnings():
       warnings.simplefilter('ignore',RuntimeWarning)
       pred=vector_pairs(array,i,j,method)
      row=summarize_predictions(pred,truth)
      row.update(phase=phase,seed=seed,missing_rate=loss,semantic_corruption=corrupt,
                 semantic_variant=variant,method=method)
      records.append(row)
sim=pd.DataFrame(records)
sim.to_csv(ROOT/'metrics'/'independent_simulation_comparison.csv',index=False)
print(sim.query("phase=='locked_test'").groupby(['semantic_variant','method'])[['coverage','error']].mean().round(4))

# Original V7 cell 31 (merged stage)
# Scenario-level aggregation; unit of replication is an independent simulated scenario (seed x condition).
check=sim.query("phase=='locked_test' and semantic_variant=='verified_semantics'")
sim_main=check.groupby('method',as_index=False).agg(
 mean_coverage=('coverage','mean'),mean_error=('error','mean'),sd_error=('error','std'),
 scenario_count=('error','count'),mean_false_direction=('false_direction_rate','mean'))
sim_main.to_csv(ROOT/'tables'/'simulation_method_summary.csv',index=False)
compare=[]
full=check.query("method=='full'").set_index(['seed','missing_rate','semantic_corruption'])
for method in [m for m in SIM_METHODS if m!='full']:
 base=check.query('method==@method').set_index(['seed','missing_rate','semantic_corruption'])
 join=full[['error','coverage']].join(base[['error','coverage']],lsuffix='_full',rsuffix='_base').dropna()
 dif=(join.error_base-join.error_full).to_numpy()
 coverage_diff=(join.coverage_full-join.coverage_base).to_numpy()
 # scenario bootstrap instead of re-sampling millions of correlated hospital pairs
 rng_ci=np.random.default_rng(42)
 ci=np.mean(dif[rng_ci.integers(0,len(dif),(2000,len(dif)))],axis=1) if len(dif)>1 else np.array([np.nan])
 compare.append(dict(baseline=method,n_scenarios=len(dif),positive_delta_means_lower_full_error=float(dif.mean()),
     delta_ci_025=float(np.nanquantile(ci,.025)),delta_ci_975=float(np.nanquantile(ci,.975)),
     coverage_difference=float(coverage_diff.mean()),
     fraction_scenarios_full_lower_error=float(np.mean(dif>0))))
paired_sim=pd.DataFrame(compare)
paired_sim.to_csv(ROOT/'bootstrap'/'simulation_paired_uncertainty.csv',index=False)
print(paired_sim.to_string(index=False))

# Original V7 cell 33 (merged stage)
# Memory-safe full-population evaluation: bounded blocks, no giant edge DataFrame.
# The initial graph exports use 350-hospital deterministic diagnostics; this stage scans ALL hospitals.
def full_scan_pairwise(X,block=192,margin=2,min_shared=3,min_agree=.60):
    A=np.asarray(X,dtype=np.float32)
    n,m=A.shape
    supported=0;eligible=0
    for i in range(n-1):
        a=A[i]
        for start in range(i+1,n,block):
            b=A[start:min(n,start+block)]
            common=np.isfinite(b)&np.isfinite(a)
            cnt=common.sum(axis=1)
            difference=np.where(common,a-b,0)
            average=difference.sum(axis=1)/np.maximum(cnt,1)
            agree=np.maximum((difference>0).sum(axis=1),(difference<0).sum(axis=1))/np.maximum(cnt,1)
            eligible+=int((cnt>=min_shared).sum())
            supported+=int(((cnt>=min_shared)&(np.abs(average)>=margin)&(agree>=min_agree)).sum())
    total=n*(n-1)//2
    return dict(hospitals=n,total_pairs=total,eligible_shared_pairs=eligible,
                supported_pairs=supported,supported_fraction=supported/total if total else np.nan)
scale_rows=[]
for year in sorted(panel.index.get_level_values(0).unique()):
    all_x=panel.loc[year].sort_index()
    for cap in [100,350,1000,len(all_x)]:
        chosen=select_hospitals(year,cap)
        t0=time.perf_counter()
        stats=full_scan_pairwise(chosen.to_numpy())
        took=time.perf_counter()-t0
        scale_rows.append(dict(year=int(year),hospital_count=len(chosen),seconds=took,
                               coverage=stats['supported_fraction'],edges=stats['supported_pairs'],
                               possible_pairs=stats['total_pairs']))
        if cap==len(all_x):
            (ROOT/'comparisons'/f'full_summary_{year}.json').write_text(json.dumps(stats,indent=2))
full_scaling=pd.DataFrame(scale_rows)
full_scaling.to_csv(ROOT/'metrics'/'all_hospital_scaling.csv',index=False)
print(full_scaling.to_string(index=False))


# Original V7 cell 35 (merged stage)
# Cross-release measurement catalog: no future release used in locked model.
catalog=pos[['Year','measure','concept','sem_key']].drop_duplicates()
mat=catalog.pivot_table(index='measure',columns='Year',values='concept',aggfunc='first')
mat.to_csv(ROOT/'qc'/'cross_release_measure_catalog.csv')
sem_dev=set(catalog[catalog.Year<=2018]['concept']);sem_test=set(catalog[catalog.Year>=2019]['concept'])
semantic_audit={'development_semantic_concepts':len(sem_dev),'test_semantic_concepts':len(sem_test),
'exact_semantic_intersection':len(sem_dev&sem_test),'test_only_semantics':len(sem_test-sem_dev),
'note':'Exact lexical alignment is a proxy, not independently validated conceptual equivalence.'}
(ROOT/'qc'/'cross_release_semantic_audit.json').write_text(json.dumps(semantic_audit,indent=2))
# TRUE composition contrast: a locked development vocabulary versus ALL available positive answer measures.
# Never mix numeric positive percentages with star ratings, linear scores or negative response categories.
allpos=pos.loc[pos.answer_num.between(0,100)].drop_duplicates(['Year','facility','concept'],keep=False)
allwide=allpos.groupby(['Year','facility'])['answer_num'].mean().rename('all_available_positive_mean')
fixed=panel.mean(axis=1).rename('locked_vocabulary_mean')
joined=pd.concat([fixed,allwide],axis=1)
rows=[]
for year,g in joined.groupby(level=0):
 g=g.droplevel(0).dropna(); delta=g.iloc[:,1]-g.iloc[:,0]
 rows.append(dict(year=int(year),n_hospitals=len(g),n_locked_concepts=len(LOCKED_CONCEPTS),
   n_available_concepts=int(allpos.loc[allpos.Year==year,'concept'].nunique()),
   spearman=float(ss.spearmanr(g.iloc[:,0],g.iloc[:,1]).statistic) if len(g)>3 and g.iloc[:,0].nunique()>1 else np.nan,
   mean_abs_difference_pp=float(delta.abs().mean()),median_abs_difference_pp=float(delta.abs().median()),
   changed_top_100=int(len(set(g.nlargest(min(100,len(g)),g.columns[0]).index) ^ set(g.nlargest(min(100,len(g)),g.columns[1]).index))),
   nonzero_difference_share=float((delta.abs()>1e-9).mean())))
rank_sensitivity=pd.DataFrame(rows)
rank_sensitivity.to_csv(ROOT/'tables'/'semantic_composition_sensitivity.csv',index=False)
joined.reset_index().to_csv(ROOT/'tables'/'locked_vs_all_available_scores.csv',index=False)
print(json.dumps(semantic_audit,indent=2));print(rank_sensitivity.to_string(index=False))

# Original V7 cell 37 (merged stage)
# Additional publication graphics; every simulated figure is explicitly labelled.
fig,ax=plt.subplots(figsize=(8,4))
for method,g in check.groupby('method'):
 z=g.groupby('missing_rate')[['error','coverage']].mean()
 ax.plot(z.index,z.error,marker='o',label=method)
ax.set(xlabel='Simulated measurement missingness',ylabel='Conditional directional error',title='Locked simulation: error under informative missingness')
ax.legend(fontsize=7,ncol=2);savefigure('F7_simulation_error_by_missingness.png')
fig,ax=plt.subplots(figsize=(7,5))
for name,g in check.groupby('method'):
 ax.scatter(g.coverage.mean(),g.error.mean(),label=name,s=55)
ax.set(xlabel='Mean supported-pair coverage',ylabel='Mean conditional directional error',title='Simulation coverage–error tradeoff')
ax.legend(fontsize=7);savefigure('F8_coverage_error_tradeoff.png')
fig,ax=plt.subplots(figsize=(8,4))
for year,g in full_scaling.groupby('year'):
 ax.plot(g.hospital_count,g.seconds,marker='o',label=str(year))
ax.set(xlabel='Hospitals compared',ylabel='Measured wall time (seconds)',title='Full-population CPU scaling');ax.legend(fontsize=8)
savefigure('F9_full_population_scaling.png')
# Explicit evidence standards for each RQ, separate descriptive outcomes from independent validation.
full_row=sim_main.query("method=='full'").iloc[0]
any_baseline=sim_main.query("method!='full'")
low_error=bool(full_row.mean_error < any_baseline.mean_error.min())
useful_coverage=bool(full_row.mean_coverage >= .25)
strong_ci=bool((paired_sim.delta_ci_025>0).all())
objectives=[
 {'objective':'O1 temporal and measure audit','status':'empirically characterized',
  'evidence':['release_audit.csv','cross_release_semantic_audit.json','temporal_overlap.csv'],
  'caveat':'Within-year reporting windows are identical; lexical semantics are only proxies.'},
 {'objective':'O2 comparability-constrained benchmarking','status':'algorithm implemented; external comparison validity unknown',
  'evidence':['all_hospital_scaling.csv','simulation_method_summary.csv'],
  'caveat':'Simulation truth is not real hospital superiority; additional external gold standard needed.'},
 {'objective':'O3 longitudinal ranking stability','status':'descriptive evidence available',
  'evidence':['longitudinal_stability.csv','semantic_composition_sensitivity.csv'],
  'caveat':'Year-to-year differences may be authentic hospital quality changes.'},
 {'objective':'O4 robustness under reporting changes','status':'controlled-simulation evaluation available',
  'evidence':['measure_loss_all.csv','independent_simulation_comparison.csv'],
  'caveat':'Artificial transformations cannot prove real-world benefit.'},
 {'objective':'O5 reproducibility and statistical validation','status':'internal evaluation available, external validation missing',
  'evidence':['hospital_cluster_bootstrap.csv','simulation_paired_uncertainty.csv'],
  'caveat':'No independent user studies or clinical policy validation.'},
]
(ROOT/'metrics'/'objective_audit.json').write_text(json.dumps(objectives,indent=2))
# Predefined decision logic; no rewriting objectives merely because numbers are inconvenient.
if low_error and useful_coverage and strong_ci:
 verdict='Simulation evidence favors the proposed method under locked test conditions. Real-world validity remains unconfirmed.'
elif low_error and useful_coverage:
 verdict='Some simulated benefits appear, but robust statistical superiority is unconfirmed.'
else:
 verdict='The main simulated superiority claim is not established. Do not claim a positive primary outcome.'
alternatives=[
 {'direction':'Measure-composition and semantic drift auditing',
  'relevant_if':'measure catalog shows real changes in concepts or definitions',
  'candidate_title':'ChronoQuality as an auditable longitudinal HCAHPS measure-compatibility system'},
 {'direction':'Ranking stability and top-k disagreement under fixed instruments',
  'relevant_if':'ranking instability is meaningfully greater than general rank correlation suggests',
  'candidate_title':'A reproducibility audit of longitudinal hospital patient-experience leaderboards'},
 {'direction':'Coverage-error benchmarking under incomplete public reporting',
  'relevant_if':'constrained method shows calibrated tradeoffs but not absolute best error',
  'candidate_title':'Risk-coverage analysis of hospital comparison policies under survey-data loss'},
]
report=f"""# ChronoQuality V2 final evidence audit

**Overall evidence verdict:** {verdict}

**Independent simulation:** full method mean error={full_row.mean_error:.5f}, coverage={full_row.mean_coverage:.5f}.
These are controlled synthetic results only.

**Data facts:** reporting windows synchronized within every release; clinical truth unavailable.

**Primary hypothesis passed?** {'Simulation only; not externally validated' if low_error and useful_coverage and strong_ci else 'No defensible pass for the primary superiority claim.'}

**Important:** Avoid editing hypotheses after inspecting held-out years. New directions are exploratory and require a separate confirmatory study.
"""
for o in objectives:report+=f"\n- **{o['objective']}**: {o['status']}; limitation: {o['caveat']}"
report+='\n\n## Exploratory alternatives, not confirmed discoveries\n'
for a in alternatives:report+=f"\n- **{a['direction']}**: {a['candidate_title']} (appropriate when {a['relevant_if']})."
(ROOT/'FINAL_EVIDENCE_AND_ALTERNATIVES.md').write_text(report)
(ROOT/'metrics'/'alternative_directions.json').write_text(json.dumps(alternatives,indent=2))
print(report)

# Original V7 cell 39 (merged stage)
print('V2 baseline stages finished; V3 additional evaluations start next. Final ZIP is created at the end.')

## V7 original — V3 — Extended independent validation, matched-coverage comparisons, and graphical evidence / V3 — Broader real-world research directions

*All original code for this stage is preserved.*

In [ ]:
# Original V7 cell 41 (merged stage)
# V3 independently generated scenario reference, explicit calibration of coverage-error tradeoffs.
from scipy.stats import wilcoxon
from itertools import product
V3_METHODS=['unmatched_mean','unmatched_median','matched_mean','majority_only','stricter_majority','full']
V3_SCENARIOS={'dev_seeds':[8101,8102,8103], 'test_seeds':list(range(99101,99113)),
              'missing_rates':[0,.15,.35,.55], 'semantic_corruption_rates':[0,.15,.35],
              'n_hospitals':500,'pairs_per_scenario':30000,'true_gap':3.0,
              'coverage_grid':[.10,.20,.30,.40,.50,.60,.70,.80]}
(ROOT/'configs'/'v3_locked_protocol.json').write_text(json.dumps(V3_SCENARIOS,indent=2))
# Score + validity masks. Margin tuned using DEV scenarios ONLY, then locked.
def method_magnitude(X,ii,jj,method):
 A=np.asarray(X,dtype=float); L=A[ii]; R=A[jj];ok=np.isfinite(L)&np.isfinite(R)
 k=ok.sum(1); diff=np.where(ok,L-R,0); d=diff.sum(1)/np.maximum(k,1)
 agree=np.maximum((diff>0).sum(1),(diff<0).sum(1))/np.maximum(k,1)
 with warnings.catch_warnings():
  warnings.simplefilter('ignore',RuntimeWarning)
  avg=np.nanmean(A,axis=1);med=np.nanmedian(A,axis=1)
 if method=='unmatched_mean':return avg[ii]-avg[jj],np.isfinite(avg[ii])&np.isfinite(avg[jj])
 if method=='unmatched_median':return med[ii]-med[jj],np.isfinite(med[ii])&np.isfinite(med[jj])
 if method=='majority_only':return d,(k>=3)&(agree>=.60)
 if method=='stricter_majority':return d,(k>=3)&(agree>=.75)
 if method=='full':return d,(k>=3)&(agree>=.60)
 if method=='matched_mean':return d,k>=3
 raise ValueError(method)
def mk_v3(seed,loss,corrupt):
 r=np.random.default_rng(seed);n=V3_SCENARIOS['n_hospitals'];p=6
 true=r.uniform(35,85,n);item=r.normal(0,4,p)
 obs=np.clip(true[:,None]+item+r.normal(0,6,(n,p)),0,100)
 # Instrument shift/coding corruption; semantic metadata assumed perfect only in oracle variant.
 bad=r.random((n,p))<corrupt;obs=np.where(bad,100-obs,obs)
 miss=r.random((n,p))<(loss+.10*(true[:,None]<50));obs[miss]=np.nan
 i=r.integers(0,n,V3_SCENARIOS['pairs_per_scenario']);j=r.integers(0,n-1,len(i));j+=(j>=i)
 truth=np.sign(true[i]-true[j]).astype('int8')
 validtruth=np.abs(true[i]-true[j])>=V3_SCENARIOS['true_gap']
 return obs,i,j,truth,validtruth
# Global margin threshold obtained from development simulations at a fixed target coverage.
# Calibration uses NO held-out scenario labels, and covers all methods fairly.
def evaluate_topk(X,i,j,truth,validtruth,method,target):
 d,eligible=method_magnitude(X,i,j,method)
 strength=np.abs(d);eligible=eligible&np.isfinite(strength)&(strength>0)
 k=min(int(__import__('builtins').round(target*len(i))),int(eligible.sum()))
 chosen=np.zeros(len(i),bool)
 if k:
  candidates=np.flatnonzero(eligible); sel=candidates[np.argpartition(strength[candidates],-k)[-k:]]
  chosen[sel]=True
 scored=chosen&validtruth
 ncorrect=int((np.sign(d[scored])==truth[scored]).sum())
 wrong=int(scored.sum())-ncorrect
 return dict(coverage=float(chosen.mean()),valid_coverage=float(scored.mean()),
   wrong=wrong,assessable=int(scored.sum()),error=float(wrong/scored.sum()) if scored.any() else np.nan,
   n_total=len(i),target_coverage=target)
records=[]
for seed,loss,corrupt in product(V3_SCENARIOS['test_seeds'],V3_SCENARIOS['missing_rates'],V3_SCENARIOS['semantic_corruption_rates']):
 X,i,j,t,validtruth=mk_v3(seed,loss,corrupt)
 for meth in V3_METHODS:
  for target in V3_SCENARIOS['coverage_grid']:
   row=evaluate_topk(X,i,j,t,validtruth,meth,target)
   records.append(dict(seed=seed,missing_rate=loss,semantic_corruption=corrupt,method=meth,**row))
v3sim=pd.DataFrame(records)
v3sim.to_csv(ROOT/'metrics'/'v3_matched_coverage_simulation.csv',index=False)
# Scenario-level aggregation avoids treating pairs from same hospitals as independent.
v3summary=v3sim.groupby(['method','target_coverage'],as_index=False).agg(
 error_mean=('error','mean'),error_sd=('error','std'),actual_coverage=('coverage','mean'),
 valid_coverage=('valid_coverage','mean'),n_scenarios=('error','count'))
v3summary.to_csv(ROOT/'tables'/'v3_error_coverage_summary.csv',index=False)
print('Independent scenarios:',v3sim[['seed','missing_rate','semantic_corruption']].drop_duplicates().shape[0],
      'method-target evaluations:',len(v3sim))

# Original V7 cell 42 (merged stage)
# Matched-coverage paired comparisons with 2000 SCENARIO-CLUSTER bootstrap draws.
# Error for unassessable zero-margin truth pairs is not labeled an error.
full=v3sim.query("method=='full'")
paired=[];gkeys=['seed','missing_rate','semantic_corruption','target_coverage']
for baseline in [m for m in V3_METHODS if m!='full']:
 joined=full.merge(v3sim.query('method==@baseline'),on=gkeys,suffixes=('_full','_base')).dropna(subset=['error_full','error_base'])
 # Differences are baseline error minus proposed error; positive favors proposed.
 for target,g in joined.groupby('target_coverage'):
  d=(g.error_base-g.error_full).to_numpy();rnd=np.random.default_rng(4200+int(100*target))
  boot=np.mean(d[rnd.integers(0,len(d),(2000,len(d)))],axis=1) if len(d)>1 else np.array([np.nan])
  paired.append(dict(baseline=baseline,target_coverage=target,n_scenarios=len(d),
       improvement_mean=float(d.mean()),ci_low=float(np.nanquantile(boot,.025)),
       ci_high=float(np.nanquantile(boot,.975)),actual_full_coverage=float(g.coverage_full.mean()),
       actual_baseline_coverage=float(g.coverage_base.mean()),
       fraction_scenarios_favor_full=float((d>0).mean())))
v3paired=pd.DataFrame(paired);v3paired.to_csv(ROOT/'bootstrap'/'v3_matched_coverage_paired_ci.csv',index=False)
print(v3paired.to_string(index=False,max_rows=16))

# Original V7 cell 44 (merged stage)
# Cross-year longitudinal composition and multi-policy robustness, hospital-balanced.
# Note: the survey count appears repeated over measure rows, so use hospital-year metadata ONCE.
from scipy.stats import kendalltau
policy_rows=[]
for year in sorted(panel.index.get_level_values(0).unique()):
 A=panel.loc[year]; nvalid=A.notna().sum(axis=1)
 policies={
 'fixed_mean':A.mean(axis=1),
 'fixed_median':A.median(axis=1),
 'fixed_trimmed':A.apply(lambda s:np.mean(np.sort(s.dropna())[1:-1]) if s.notna().sum()>3 else s.mean(),axis=1),
 'all_available_positive_mean':allwide.loc[year].reindex(A.index)
 }
 ref=policies['fixed_mean']
 for name,v in policies.items():
  g=pd.concat([ref,v],axis=1).dropna(); a=g.iloc[:,0];b=g.iloc[:,1]
  topa=set(a.nlargest(min(100,len(a))).index);topb=set(b.nlargest(min(100,len(b))).index)
  policy_rows.append(dict(year=int(year),policy=name,n_hospitals=len(g),
     spearman=float(spearmanr(a,b).statistic) if a.nunique()>1 and b.nunique()>1 else np.nan,
     kendall=float(kendalltau(a,b).statistic) if a.nunique()>1 and b.nunique()>1 else np.nan,
     top100_jaccard=float(len(topa&topb)/max(1,len(topa|topb))),
     absolute_mean_pp=float(np.abs(a-b).mean()),
     ranking_policy='Descriptive score sensitivity, NOT independent truth'))
policies=pd.DataFrame(policy_rows);policies.to_csv(ROOT/'tables'/'v3_real_scoring_policy_sensitivity.csv',index=False)
# Temporal turnover: changes for matched hospitals, each release compared by SAME locked vocabulary.
turn=[]
for ya,yb in [(2016,2017),(2017,2018),(2018,2019),(2019,2020)]:
 pa=panel.loc[ya].mean(axis=1);pb=panel.loc[yb].mean(axis=1)
 idx=pa.index.intersection(pb.index);delta=(pb.loc[idx]-pa.loc[idx]).dropna()
 turn.extend([dict(year_pair=f'{ya}-{yb}',facility=str(x),score_change_pp=float(v)) for x,v in delta.items()])
turn=pd.DataFrame(turn);turn.to_csv(ROOT/'tables'/'v3_matched_hospital_year_changes.csv',index=False)
print(policies.to_string(index=False,max_rows=24))

# Original V7 cell 45 (merged stage)
# Additional unambiguous figures (300 dpi) + manifest of quantitative plots.
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size':10,'axes.spines.top':False,'axes.spines.right':False})
def v3fig(name):
 plt.tight_layout();plt.savefig(ROOT/'figures'/name,dpi=300,bbox_inches='tight');plt.close()
fig,ax=plt.subplots(figsize=(9,5))
for method,g in v3summary.groupby('method'):
 ax.plot(g.actual_coverage,g.error_mean*100,marker='o',label=method)
ax.set(xlabel='Actual supported pair coverage',ylabel='Conditional directional error (%)',title='Independent simulation: error at matched coverage');ax.legend(fontsize=8,ncol=2)
v3fig('V3_F10_matched_coverage_error.png')
fig,ax=plt.subplots(figsize=(9,5))
z=v3paired[v3paired.target_coverage==.4].sort_values('improvement_mean')
ax.errorbar(z.improvement_mean*100,np.arange(len(z)),xerr=[(z.improvement_mean-z.ci_low)*100,(z.ci_high-z.improvement_mean)*100],fmt='o',capsize=3)
ax.axvline(0,color='grey',linestyle='--');ax.set(yticks=np.arange(len(z)),yticklabels=z.baseline,xlabel='Baseline error − ChronoQuality error (percentage points)',title='Matched 40% target coverage: paired scenario 95% intervals')
v3fig('V3_F11_paired_effect_forest.png')
fig,ax=plt.subplots(figsize=(8,5))
for name,g in v3sim.groupby('method'):
 u=g[g.target_coverage==.4].groupby('missing_rate').error.mean()
 ax.plot(u.index,u.values*100,marker='o',label=name)
ax.set(xlabel='Injected missingness fraction',ylabel='Conditional directional error (%)',title='Synthetic measurement loss, nominal 40% pair coverage');ax.legend(fontsize=7,ncol=2)
v3fig('V3_F12_error_vs_missingness.png')
fig,ax=plt.subplots(figsize=(8,5))
for name,g in v3sim.groupby('method'):
 u=g[g.target_coverage==.4].groupby('semantic_corruption').error.mean()
 ax.plot(u.index,u.values*100,marker='o',label=name)
ax.set(xlabel='Injected semantic corruption fraction',ylabel='Conditional directional error (%)',title='Synthetic semantic corruption at nominal 40% pair coverage');ax.legend(fontsize=7,ncol=2)
v3fig('V3_F13_error_vs_semantic_shift.png')
fig,ax=plt.subplots(figsize=(8,4))
q=rank_sensitivity.set_index('year');ax.bar(q.index.astype(str),q.mean_abs_difference_pp)
ax.set(xlabel='Release year',ylabel='Mean absolute difference (percentage points)',title='Observed: locked versus all-available positive-measure scores')
v3fig('V3_F14_real_measure_composition.png')
fig,ax=plt.subplots(figsize=(8,5));p=policies.pivot(index='year',columns='policy',values='top100_jaccard')
im=ax.imshow(p.values,aspect='auto',vmin=0,vmax=1);ax.set_xticks(np.arange(len(p.columns)),p.columns,rotation=25,ha='right');ax.set_yticks(np.arange(len(p.index)),p.index);fig.colorbar(im,ax=ax,label='Top-100 Jaccard')
ax.set_title('Observed top-100 agreement with locked mean');v3fig('V3_F15_policy_top100_heatmap.png')
fig,ax=plt.subplots(figsize=(9,4)); groups=[x.score_change_pp.values for _,x in turn.groupby('year_pair')];labels=list(turn.year_pair.unique());ax.boxplot(groups,tick_labels=labels,showfliers=False)
ax.axhline(0,color='grey',linestyle='--');ax.set(ylabel='Matched-hospital change (pp)',xlabel='Successive releases',title='Observed longitudinal score changes (fixed measure vocabulary)')
v3fig('V3_F16_real_longitudinal_change.png')
fig,ax=plt.subplots(figsize=(9,5));t=v3paired.pivot(index='target_coverage',columns='baseline',values='improvement_mean');
for name in t.columns:ax.plot(t.index,100*t[name],marker='o',label=name)
ax.axhline(0,color='grey',linestyle='--');ax.set(xlabel='Requested pair coverage',ylabel='Baseline minus proposed error (pp)',title='Comparison advantage varies with coverage');ax.legend(fontsize=7,ncol=2)
v3fig('V3_F17_advantage_over_coverage.png')
print('V3 figures:',len(list((ROOT/'figures').glob('V3_*.png'))))

# Original V7 cell 46 (merged stage)
# Predeclared claims audit — no selecting a favorable coverage point post hoc.
# Primary comparison at requested 40% coverage with 95% cluster-scene confidence interval.
at40=v3paired[np.isclose(v3paired.target_coverage,.4)]
all_lower=bool(len(at40)>0 and (at40.ci_low>0).all())
strongest=at40.loc[at40.improvement_mean.idxmin()].to_dict() if len(at40) else {}
nontrivial=bool((rank_sensitivity.mean_abs_difference_pp.fillna(0)>0.01).any())
conclusions={
 'O1_measure_compatibility':'Descriptively characterized; clinical semantic equivalence not adjudicated',
 'O2_partial_benchmarking':'Implemented; superiority at nominal 40% coverage '+('supported in independent simulation against all listed baselines' if all_lower else 'NOT established against all baselines'),
 'O3_longitudinal_stability':'Characterized for matched hospital IDs; changes not causally attributed',
 'O4_robustness':'Synthetic perturbation evaluated; do not extrapolate to actual clinical misranking',
 'O5_reproducibility':'Code, QC, scenario-level uncertainty, output inventory; no external clinical validation',
 'composition_effect_nonzero':nontrivial,
 'all_baseline_95CI_advantage_at_40pct':all_lower,
 'weakest_baseline_comparison':strongest,
 'limitation':'Five HCAHPS releases only; common window within year; aggregates lack independent hospital quality truth'
}
(ROOT/'metrics'/'v3_objective_evidence_audit.json').write_text(json.dumps(conclusions,indent=2,default=str))
alts=[
 dict(title='Auditable measure-composition drift in longitudinal patient-experience benchmarks',
 condition='Observed nontrivial all-available versus fixed-vocabulary score differences',
 evidence='v3_real_scoring_policy_sensitivity.csv and semantic_composition_sensitivity.csv',
 qualification='Measure change cannot be isolated from clinical/response change without additional identification'),
 dict(title='Coverage–error frontiers for partial hospital orderings under corrupted measurement systems',
 condition='Controlled scenarios show non-dominated policies at matched coverage',
 evidence='v3_matched_coverage_simulation.csv and v3_matched_coverage_paired_ci.csv',
 qualification='Simulation-only methodology; not validated clinical hospital superiority'),
 dict(title='Top-k leaderboard instability versus population-wide rank correlation',
 condition='Top-100 overlap varies materially despite high general correlation',
 evidence='longitudinal_stability.csv and v3_real_scoring_policy_sensitivity.csv',
 qualification='Descriptive findings; avoid asserting why rankings changed')]
(ROOT/'metrics'/'v3_candidate_objectives.json').write_text(json.dumps(alts,indent=2))
report=['# ChronoQuality V3 — Evidence and research directions','',
'**These findings do not guarantee any objective is achieved or journal acceptance.**','',
'## Prespecified objective audit','']
for k,v in conclusions.items():report.append(f'- **{k}:** {v}')
report+=['','## Candidate refinements (exploratory, not confirmed innovations)','']
for row in alts:report += [f"### {row['title']}",f"Trigger: {row['condition']}",f"Evidence: {row['evidence']}",f"Qualification: {row['qualification']}",'']
report+=['## Interpretation guardrails','- Comparison error is relative to latent simulated ranking, not true patient-care quality.',
'- Whole-cohort ranking is not identifiable from a few aggregate survey indicators.',
'- Source survey sample size is not the denominator for each question response category.',
'- Locking development vocabulary prevents test-release leakage for the fixed-vocabulary model.',
'- Full-catalog analyses are explicitly post hoc/descriptive and must not be reported as locked-test predictions.']
(ROOT/'FINAL_V3_RESEARCH_EVIDENCE.md').write_text('\n'.join(report))
print('\n'.join(report[:15]))

# Original V7 cell 47 (merged stage)
# Final artifact validation and comprehensive ZIP export (the only required V3 archive).
critical=[ROOT/'metrics'/'v3_matched_coverage_simulation.csv',ROOT/'bootstrap'/'v3_matched_coverage_paired_ci.csv',
 ROOT/'tables'/'semantic_composition_sensitivity.csv',ROOT/'tables'/'v3_real_scoring_policy_sensitivity.csv',
 ROOT/'metrics'/'v3_objective_evidence_audit.json',ROOT/'FINAL_V3_RESEARCH_EVIDENCE.md']
assert all(p.exists() and p.stat().st_size>100 for p in critical),[str(p) for p in critical if not p.exists()]
assert v3sim.groupby(['seed','missing_rate','semantic_corruption','method']).size().eq(len(V3_SCENARIOS['coverage_grid'])).all()
assert len(list((ROOT/'figures').glob('V3_*.png')))>=7
artifacts=[]
for f in sorted(ROOT.rglob('*')):
 if f.is_file():artifacts.append({'path':str(f.relative_to(ROOT)),'size_bytes':f.stat().st_size})
(ROOT/'qc'/'v3_final_file_manifest.json').write_text(json.dumps(artifacts,indent=2))
out=Path('/kaggle/working/ChronoQuality_RUN_003_Results.zip')
with zipfile.ZipFile(out,'w',zipfile.ZIP_DEFLATED,compresslevel=4,allowZip64=True) as z:
 for f in sorted(ROOT.rglob('*')):
  if f.is_file():z.write(f,arcname=str(f.relative_to(ROOT.parent)))
print('V3 OUTPUT ZIP:',out,'MB:',__import__('builtins').round(out.stat().st_size/1048576,2),'FILES:',len(artifacts))

## V7 original — V4.0 — Scope safeguards and traceable comparison design / V4.1 — Common-hospital/common-measure analyses, release catalogs and real score-policy differences

*All original code for this stage is preserved.*

In [ ]:
# Original V7 cell 49 (merged stage)
import itertools, warnings
from scipy.stats import spearmanr, kendalltau
V4=ROOT/'v4'; V4.mkdir(exist_ok=True)
for d in ['tables','figures','bootstrap','qc','metrics','counterfactual','subgroups']:(V4/d).mkdir(exist_ok=True)
V4_CFG={'seed':42,'status':'exploratory_post_V3','top_k':[10,25,50,100,250,500],
        'bootstrap_hospitals':500,'policy_weights':[0.0,0.25,0.5,0.75,1.0],
        'perturbation_rates':[0.0,0.1,0.2,0.4], 'perturbation_repetitions':20,
        'min_concepts':3,'stable_text_equivalence':'exact_normalized_text_only',
        'policy_family':['mean','median','trimmed','all_available_mean','hybrid']}
(V4/'qc'/'config.json').write_text(json.dumps(V4_CFG,indent=2))
assert set([2016,2017,2018,2019,2020]).issubset(set(panel.index.get_level_values(0)))
print('V4 running on real panel:',panel.shape, 'positive concepts:',len(panel.columns))

# Original V7 cell 51 (merged stage)
def topk_set(s,k):
    # deterministic tie break prevents spurious nondeterminism
    s=s.dropna();return set(s.sort_values(ascending=False,kind='stable').index[:min(k,len(s))])
def jaccard(a,b):return len(a&b)/max(1,len(a|b))
def safe_corr(a,b):
    g=pd.concat([a,b],axis=1).dropna()
    return float(spearmanr(g.iloc[:,0],g.iloc[:,1]).statistic) if len(g)>3 and g.iloc[:,0].nunique()>1 and g.iloc[:,1].nunique()>1 else np.nan
all_years=sorted(panel.index.get_level_values(0).unique().tolist())
common_hospitals=set.intersection(*[set(panel.loc[y].index) for y in all_years])
common_concepts=set(panel.columns[panel.notna().groupby(level=0).sum().min(axis=0)>0])
# This is a *fully balanced* universe of identifiers; restrict further to rows with score evidence.
conf=pd.DataFrame([{'year':y,'hospital_year_rows':len(panel.loc[y]),'concepts_nonempty':int(panel.loc[y].notna().any().sum()),
                    'common_hospitals_five_years':len(common_hospitals),'common_concepts_five_years':len(common_concepts),
                    'reported_window_count':int(raw.loc[raw.Year==y,['Start Date','End Date']].drop_duplicates().shape[0])} for y in all_years])
conf.to_csv(V4/'tables'/'panel_feasibility.csv',index=False)
# Score variations do not combine star rating, linear and percentage measures.
policy_scores={}
for year in all_years:
    X=panel.loc[year].astype(float)
    means=X.mean(axis=1); med=X.median(axis=1)
    vals=np.sort(X.to_numpy(dtype=float),axis=1)
    # NaNs sort to end; row-wise trimmed mean over nonmissing observations
    trimmed=X.apply(lambda r: float(np.mean(np.sort(r.dropna().to_numpy())[1:-1])) if r.notna().sum()>3 else float(r.mean()),axis=1)
    ext=allwide.loc[year].reindex(X.index)
    policy_scores[year]=pd.DataFrame({'fixed_mean':means,'fixed_median':med,'fixed_trimmed':trimmed,
                                     'all_available_mean':ext,'hybrid_mean_median':0.5*(means+med)},index=X.index)
policy_long=pd.concat(policy_scores,names=['Year','facility'])
policy_long.to_csv(V4/'tables'/'hospital_scoring_policies.csv')
# A score policy changes the mathematical definition of a benchmark; not a change in clinical outcome.
policy_diff=[]
for year,g in policy_scores.items():
    for col in g.columns:
        if col=='fixed_mean':continue
        z=g[['fixed_mean',col]].dropna()
        row={'year':year,'policy':col,'n':len(z),'rho':safe_corr(z.fixed_mean,z[col]),
             'mean_absolute_score_shift':float((z.fixed_mean-z[col]).abs().mean())}
        for k in V4_CFG['top_k']:
            if len(z)>=k:row[f'top_{k}_jaccard']=jaccard(topk_set(z.fixed_mean,k),topk_set(z[col],k))
        policy_diff.append(row)
policy_diff=pd.DataFrame(policy_diff);policy_diff.to_csv(V4/'tables'/'within_year_policy_instability.csv',index=False)
print(conf.to_string(index=False));print(policy_diff.head(10).to_string(index=False))

# Original V7 cell 53 (merged stage)
stability=[];trajectory=[]
for ya,yb in zip(all_years[:-1],all_years[1:]):
    A=policy_scores[ya]; B=policy_scores[yb]; shared=A.index.intersection(B.index)
    for policy in A.columns.intersection(B.columns):
        a=A.loc[shared,policy];b=B.loc[shared,policy];good=a.notna() & b.notna();a=a[good];b=b[good]
        if len(a)<20:continue
        ra=a.rank(ascending=False,method='average');rb=b.rank(ascending=False,method='average')
        st={'year_a':ya,'year_b':yb,'policy':policy,'n_shared':len(a),'rho':safe_corr(a,b),
            'mean_absolute_rank_movement':float((ra-rb).abs().mean()),
            'median_absolute_rank_movement':float((ra-rb).abs().median())}
        for k in V4_CFG['top_k']:
            if len(a)>=k:st[f'top{k}_jaccard']=jaccard(topk_set(a,k),topk_set(b,k))
        stability.append(st)
        if policy=='fixed_mean':
            for facility in shared[good]:trajectory.append({'hospital':str(facility),'years':f'{ya}-{yb}',
               'score_a':float(a.loc[facility]),'score_b':float(b.loc[facility]),
               'rank_a':float(ra.loc[facility]),'rank_b':float(rb.loc[facility]),
               'rank_movement':float(rb.loc[facility]-ra.loc[facility])})
stability=pd.DataFrame(stability);trajectory=pd.DataFrame(trajectory)
stability.to_csv(V4/'tables'/'multik_longitudinal_stability.csv',index=False)
trajectory.to_csv(V4/'tables'/'hospital_rank_trajectories.csv',index=False)
print(stability.head().to_string(index=False))

# Original V7 cell 55 (merged stage)
decomp=[];details=[]
for ya,yb in zip(all_years[:-1],all_years[1:]):
    idx=policy_scores[ya].index.intersection(policy_scores[yb].index)
    a=policy_scores[ya].loc[idx];b=policy_scores[yb].loc[idx]
    # all-available difference = fixed-score change + composition difference at later period - earlier period
    q=pd.DataFrame({'fixed_a':a.fixed_mean,'fixed_b':b.fixed_mean,
        'all_a':a.all_available_mean,'all_b':b.all_available_mean}).dropna()
    if q.empty:continue
    q['observed_all_change']=q.all_b-q.all_a
    q['fixed_performance_change']=q.fixed_b-q.fixed_a
    q['composition_shift_later']=q.all_b-q.fixed_b
    q['composition_shift_earlier']=-(q.all_a-q.fixed_a)
    q['reconstruction_error']=q.observed_all_change-(q.fixed_performance_change+q.composition_shift_later+q.composition_shift_earlier)
    assert q.reconstruction_error.abs().max()<1e-8
    q['comparison']=f'{ya}-{yb}'
    details.append(q.assign(facility=q.index).reset_index(drop=True))
    parts=q[['fixed_performance_change','composition_shift_later','composition_shift_earlier']]
    row={'comparison':f'{ya}-{yb}','n':len(q),'mean_abs_total_change':float(q.observed_all_change.abs().mean()),
         'mean_abs_fixed_change':float(parts.fixed_performance_change.abs().mean()),
         'mean_abs_later_composition':float(parts.composition_shift_later.abs().mean()),
         'mean_abs_earlier_composition':float(parts.composition_shift_earlier.abs().mean()),
         'share_nonzero_composition':float(((parts.composition_shift_later+parts.composition_shift_earlier).abs()>1e-8).mean()),
         'max_reconstruction_error':float(q.reconstruction_error.abs().max())}
    decomp.append(row)
decomp=pd.DataFrame(decomp);decomp_detail=pd.concat(details,ignore_index=True)
decomp.to_csv(V4/'tables'/'score_change_decomposition.csv',index=False)
decomp_detail.to_csv(V4/'tables'/'hospital_level_score_decomposition.csv',index=False)
print(decomp.to_string(index=False))

# Original V7 cell 57 (merged stage)
shap=[]
for ya,yb in zip(all_years[:-1],all_years[1:]):
    idx=policy_scores[ya].index.intersection(policy_scores[yb].index)
    for policy in ['fixed_median','all_available_mean','fixed_trimmed']:
        a=policy_scores[ya].loc[idx,['fixed_mean',policy]];b=policy_scores[yb].loc[idx,['fixed_mean',policy]]
        keep=a.notna().all(axis=1)&b.notna().all(axis=1);a=a[keep];b=b[keep]
        if len(a)<100:continue
        for k in [25,100,250]:
            if len(a)<k:continue
            # value = top-k overlap relative to baseline top-k from year a, fixed mean
            reference=topk_set(a.fixed_mean,k)
            f00=jaccard(reference,topk_set(a.fixed_mean,k))
            f10=jaccard(reference,topk_set(b.fixed_mean,k))
            f01=jaccard(reference,topk_set(a[policy],k))
            f11=jaccard(reference,topk_set(b[policy],k))
            year_effect=.5*((f10-f00)+(f11-f01))
            policy_effect=.5*((f01-f00)+(f11-f10))
            assert abs(year_effect+policy_effect-(f11-f00))<1e-10
            shap.append({'year_a':ya,'year_b':yb,'policy':policy,'k':k,'n':len(a),
                  'baseline_topk_overlap':f00,'combined_topk_overlap':f11,
                  'year_attribution':year_effect,'policy_attribution':policy_effect,
                  'total_change':f11-f00})
shap=pd.DataFrame(shap);shap.to_csv(V4/'tables'/'two_factor_shapley_topk.csv',index=False)
print(shap.head(12).to_string(index=False))

## V7 original — V4.5 — Policy robustness and uncertainty over defensible score specifications / V4.6 — Missing-measure stress tests, leave-one-measure-out and matched hospital subsets

*All original code for this stage is preserved.*

In [ ]:
# Original V7 cell 59 (merged stage)
robust=[];members=[]
for year,g in policy_scores.items():
    gg=g.dropna(); n=len(gg)
    if n<100:continue
    R=gg.rank(ascending=False,method='average')
    membership={}
    for k in V4_CFG['top_k']:
        if k>n:continue
        indicator=pd.DataFrame({c:gg.index.isin(topk_set(gg[c],k)).astype(int) for c in gg.columns},index=gg.index)
        membership[k]=indicator.mean(axis=1)
        robust.append({'year':year,'k':k,'hospitals':n,'stable_in_all_policies':int((membership[k]==1).sum()),
            'in_some_policies':int((membership[k]>0).sum()),'borderline_between_policies':int(((membership[k]>0)&(membership[k]<1)).sum()),
            'stable_fraction_of_k':float((membership[k]==1).sum()/k)})
    for facility in gg.index:
        item={'year':year,'facility':str(facility),'rank_min':float(R.loc[facility].min()),
              'rank_max':float(R.loc[facility].max()),'rank_range':float(R.loc[facility].max()-R.loc[facility].min())}
        for k,member in membership.items():item[f'top{k}_frequency']=float(member.loc[facility])
        members.append(item)
robust=pd.DataFrame(robust);members=pd.DataFrame(members)
robust.to_csv(V4/'tables'/'robust_topk_membership.csv',index=False)
members.to_csv(V4/'tables'/'hospital_rank_policy_uncertainty.csv',index=False)
print(robust.to_string(index=False,max_rows=30))

# Original V7 cell 61 (merged stage)
stress=[];loo=[];RNG=np.random.default_rng(404)
for year in all_years:
    X=panel.loc[year].to_numpy(dtype=float); idx=panel.loc[year].index
    original=pd.Series(np.nanmean(X,axis=1),index=idx)
    for j,concept in enumerate(panel.columns):
        Y=X.copy();Y[:,j]=np.nan
        with warnings.catch_warnings():
            warnings.simplefilter('ignore',RuntimeWarning)
            new=pd.Series(np.nanmean(Y,axis=1),index=idx)
        z=pd.concat([original,new],axis=1).dropna()
        if len(z)>100:loo.append({'year':year,'removed_concept':str(concept),'n':len(z),
          'rho':safe_corr(z.iloc[:,0],z.iloc[:,1]),'top100_jaccard':jaccard(topk_set(z.iloc[:,0],100),topk_set(z.iloc[:,1],100))})
    for rate in V4_CFG['perturbation_rates']:
        for rep in range(V4_CFG['perturbation_repetitions']):
            Y=X.copy()
            if rate:
                # hospital-specific missingness: reproducible synthetic, not genuine reporting shifts
                drop=RNG.random(Y.shape)<rate; Y[drop]=np.nan
            with warnings.catch_warnings():
                warnings.simplefilter('ignore',RuntimeWarning)
                nscore=pd.Series(np.nanmean(Y,axis=1),index=idx)
            z=pd.concat([original,nscore],axis=1).dropna()
            if len(z)<100:continue
            stress.append({'year':year,'rate':rate,'repeat':rep,'n_evaluable':len(z),
                           'rho':safe_corr(z.iloc[:,0],z.iloc[:,1]),
                           'top100_jaccard':jaccard(topk_set(z.iloc[:,0],100),topk_set(z.iloc[:,1],100)),
                           'fraction_without_numeric_score':float(nscore.isna().mean())})
stress=pd.DataFrame(stress);loo=pd.DataFrame(loo)
stress.to_csv(V4/'tables'/'simulated_missing_measure_stress.csv',index=False)
loo.to_csv(V4/'tables'/'leave_one_measure_out.csv',index=False)
print(stress.groupby('rate')[['rho','top100_jaccard']].mean().to_string())

# Original V7 cell 63 (merged stage)
brng=np.random.default_rng(4242); boot=[]; B=V4_CFG['bootstrap_hospitals']
for year,g in policy_scores.items():
    g=g[['fixed_mean','all_available_mean']].dropna(); n=len(g)
    if n<200:continue
    arr=g.to_numpy(dtype=float)
    for k in [25,100,250]:
        res=[]
        for rep in range(B):
            sampled=arr[brng.integers(n,size=n)]
            # replica indices ensure bootstrap duplicates are distinguishable
            a=pd.Series(sampled[:,0]);b=pd.Series(sampled[:,1])
            res.append(jaccard(topk_set(a,k),topk_set(b,k)))
        boot.append({'year':year,'k':k,'B':B,'observed':jaccard(topk_set(g.fixed_mean,k),topk_set(g.all_available_mean,k)),
           'bootstrap_mean':float(np.mean(res)),'ci_low':float(np.percentile(res,2.5)),
           'ci_high':float(np.percentile(res,97.5))})
boot=pd.DataFrame(boot);boot.to_csv(V4/'bootstrap'/'policy_topk_hospital_cluster_bootstrap.csv',index=False)
print(boot.to_string(index=False))

# Original V7 cell 65 (merged stage)
checks=[]; neg_rng=np.random.default_rng(901)
for year,g in policy_scores.items():
    z=g[['fixed_mean','all_available_mean']].dropna();a=z.fixed_mean;b=z.all_available_mean
    if len(z)<250:continue
    perm=pd.Series(b.to_numpy()[neg_rng.permutation(len(b))],index=b.index)
    for k in [25,100,250]:
        checks.append({'year':year,'k':k,'identity_jaccard':jaccard(topk_set(a,k),topk_set(a.copy(),k)),
          'actual_policy_jaccard':jaccard(topk_set(a,k),topk_set(b,k)),
          'shuffled_id_placebo_jaccard':jaccard(topk_set(a,k),topk_set(perm,k))})
checks=pd.DataFrame(checks)
assert (checks.identity_jaccard==1).all()
assert (decomp.max_reconstruction_error<1e-8).all()
checks.to_csv(V4/'qc'/'invariance_and_placebo_tests.csv',index=False)
print(checks.to_string(index=False))

# Original V7 cell 67 (merged stage)
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
plt.rcParams.update({'font.size':10,'axes.spines.top':False,'axes.spines.right':False,'savefig.bbox':'tight'})
FIG=V4/'figures'
def figsave(fig,name):
    for fmt in ('png','svg','pdf'):fig.savefig(FIG/f'{name}.{fmt}',dpi=300,bbox_inches='tight')
    plt.close(fig)
# Figure 1: measurement-availability landscape
T=pos.groupby(['Year','measure'])['answer_num'].apply(lambda x:x.between(0,100).mean()).unstack(fill_value=0)
T=T.reindex(index=all_years)
fig,ax=plt.subplots(figsize=(15,4)); im=ax.imshow(T.values,aspect='auto',vmin=0,vmax=1,cmap='viridis');ax.set_yticks(range(len(T.index)),T.index);ax.set_xticks([]);ax.set_xlabel('HCAHPS measure IDs (ordered)');ax.set_ylabel('Release year');ax.set_title('Observed fraction with numeric answers by measure');fig.colorbar(im,ax=ax,label='Available fraction');figsave(fig,'F01_measure_availability_heatmap')
# Figure 2: policy rank agreement matrix in latest release
z=policy_scores[2020].dropna();fig,ax=plt.subplots(figsize=(7,6));C=z.corr(method='spearman'); im=ax.imshow(C.values,vmin=.75,vmax=1,cmap='magma');ax.set_xticks(range(len(C)),C.columns,rotation=45,ha='right');ax.set_yticks(range(len(C)),C.index);ax.set_title('Within-year score-policy rank agreement (2020)');fig.colorbar(im,ax=ax);figsave(fig,'F02_policy_rank_agreement')
# Figure 3: multi-k top-set similarity
fig,ax=plt.subplots(figsize=(9,5));latest=policy_diff[policy_diff.year==2020]
for _,r in latest.iterrows():ax.plot(V4_CFG['top_k'],[r.get(f'top_{k}_jaccard',np.nan) for k in V4_CFG['top_k']],marker='o',label=r.policy)
ax.set(xlabel='Top-k cutoff',ylabel='Jaccard agreement',title='Top-k sensitivity across scoring specifications (2020)',ylim=(0,1.05));ax.legend(fontsize=8);figsave(fig,'F03_multik_policy_instability')
# Figure 4: longitudinal top-k
fig,ax=plt.subplots(figsize=(9,5));ss=stability[stability.policy=='fixed_mean']
for _,r in ss.iterrows():ax.plot(V4_CFG['top_k'],[r.get(f'top{k}_jaccard',np.nan) for k in V4_CFG['top_k']],marker='o',label=f'{r.year_a}-{r.year_b}')
ax.set(xlabel='Top-k cutoff',ylabel='Jaccard agreement',title='Longitudinal top-k turnover: common hospitals and fixed measure vocabulary',ylim=(0,1.05));ax.legend();figsave(fig,'F04_multik_longitudinal')
# Figure 5: score-change decomposition (absolute contribution magnitude; NOT additive shares)
fig,ax=plt.subplots(figsize=(9,5));cols=['mean_abs_fixed_change','mean_abs_later_composition','mean_abs_earlier_composition']; x=np.arange(len(decomp));width=.25
for j,c in enumerate(cols):ax.bar(x+(j-1)*width,decomp[c],width,label=c.replace('mean_abs_','').replace('_',' '))
ax.set_xticks(x,decomp.comparison);ax.set(ylabel='Mean absolute shift (percentage points)',title='Observable score-shift components (magnitudes not additive)');ax.legend(fontsize=8);figsave(fig,'F05_score_shift_decomposition')
# Figure 6: factor-attribution waterfall style
fig,ax=plt.subplots(figsize=(10,5));s=shap[(shap.policy=='all_available_mean')&(shap.k==100)].copy();xx=np.arange(len(s));ax.bar(xx-.2,s.year_attribution,.4,label='Release-year attribution');ax.bar(xx+.2,s.policy_attribution,.4,label='Policy attribution');ax.axhline(0,color='k',lw=.8);ax.set_xticks(xx,[f'{a}-{b}' for a,b in zip(s.year_a,s.year_b)]);ax.set(ylabel='Shapley contribution to Jaccard change',title='Two-factor decomposition of top-100 turnover');ax.legend();figsave(fig,'F06_shapley_top100')
# Figure 7: robust membership bands
fig,ax=plt.subplots(figsize=(9,5));r=robust.pivot(index='k',columns='year',values='stable_fraction_of_k');r.plot(ax=ax,marker='o');ax.set(xlabel='Top-k cutoff',ylabel='Fraction stable in every scoring policy',title='Ranking-policy robust top-k membership',ylim=(0,1.05));figsave(fig,'F07_robust_membership')
# Figure 8: stress ribbons across simulation repetitions
fig,ax=plt.subplots(figsize=(9,5));g=stress.groupby('rate').top100_jaccard.agg(['mean','std']);x=g.index.values.astype(float);ax.plot(x,g['mean'].values,marker='o',label='Mean');ax.fill_between(x,np.clip(g['mean']-g['std'],0,1),np.clip(g['mean']+g['std'],0,1),alpha=.2,label='±1 scenario SD');ax.set(xlabel='Simulated independent cell removal rate',ylabel='Top-100 overlap',title='Synthetic data-availability stress test',ylim=(0,1.05));ax.legend();figsave(fig,'F08_synthetic_missingness_stress')
# Figure 9: bootstrap intervals
fig,ax=plt.subplots(figsize=(10,6));b=boot[boot.k==100].sort_values('year'); yy=np.arange(len(b));ax.errorbar(b.bootstrap_mean,yy,xerr=[b.bootstrap_mean-b.ci_low,b.ci_high-b.bootstrap_mean],fmt='o',capsize=4);ax.scatter(b.observed,yy,marker='x',label='Observed');ax.legend();ax.set_yticks(yy,b.year.astype(str));ax.set(xlabel='Top-100 Jaccard',title='Policy top-100 overlap: hospital bootstrap percentile intervals',xlim=(0,1.05));figsave(fig,'F09_bootstrap_forest')
# Figure 10: hospital-specific ranked movement dispersion
fig,ax=plt.subplots(figsize=(10,5));groups=[trajectory.loc[trajectory.years==y,'rank_movement'].clip(-1000,1000).values for y in sorted(trajectory.years.unique())];ax.violinplot(groups,showmedians=True);ax.set_xticks(range(1,len(groups)+1),sorted(trajectory.years.unique()));ax.set(ylabel='Relative rank movement (clipped to ±1,000 for display)',title='Distribution of observed rank movements');figsave(fig,'F10_rank_movement_violin')
# Figure 11: actual policy vs negative-control overlap
fig,ax=plt.subplots(figsize=(9,5));q=checks[checks.k==100];xx=np.arange(len(q));ax.plot(xx,q.actual_policy_jaccard,'o-',label='Observed policy overlap');ax.plot(xx,q.shuffled_id_placebo_jaccard,'s--',label='Shuffled-identifier placebo');ax.set_xticks(xx,q.year.astype(str));ax.set(ylabel='Top-100 Jaccard',title='Observed ranking overlap versus identifier-placebo',ylim=(0,1.05));ax.legend();figsave(fig,'F11_placebo_test')
# Figure 12: leave-one-measure-out instability
fig,ax=plt.subplots(figsize=(9,5));d=loo.groupby('year').top100_jaccard.agg(['mean','min','max']);xx=np.arange(len(d));ax.errorbar(xx,d['mean'],yerr=[d['mean']-d['min'],d['max']-d['mean']],fmt='o',capsize=4);ax.set_xticks(xx,d.index.astype(str));ax.set(ylabel='Top-100 Jaccard vs full-score baseline',title='Leave-one-measure-out score-policy sensitivity',ylim=(0,1.05));figsave(fig,'F12_leave_one_measure_out')
print('V4 publication graphics saved:',len(list(FIG.glob('*.png'))),'PNG plus SVG/PDF counterparts')

# Original V7 cell 69 (merged stage)
# Audit gates depend on values generated by this run; no preprogrammed favorable conclusions.
audits={
 'O1_measure_catalog_and_temporal_compatibility':{'status':'SUPPORTED_DESCRIPTIVELY','evidence':'panel_feasibility.csv; earlier semantic catalog audit','limitations':'Exact normalized text ≠ expert-reviewed clinical equivalence.'},
 'O2_rank_change_decomposition':{'status':'SUPPORTED_AS_ACCOUNTING_IDENTITY','evidence':'hospital_level_score_decomposition.csv; two_factor_shapley_topk.csv','limitations':'Effects are descriptive analytical attributions, not causal drivers of hospital care.'},
 'O3_multik_instability':{'status':'SUPPORTED_DESCRIPTIVELY','evidence':'multik_longitudinal_stability.csv; within_year_policy_instability.csv','limitations':'No independent hospital quality reference.'},
 'O4_robust_membership':{'status':'SUPPORTED_AS_POLICY_SENSITIVITY','evidence':'hospital_rank_policy_uncertainty.csv; robust_topk_membership.csv','limitations':'Policy variation is not patient-level sampling uncertainty.'},
 'O5_validation_and_reproducibility':{'status':'PARTIAL','evidence':'bootstrap, perturbations, negative controls, code/manifest','limitations':'Exploratory post V3; no untouched external confirmation or prospective operational evaluation.'},
 'methodological_superiority':{'status':'UNESTABLISHED','evidence':'V3 and V4 baselines','limitations':'Ranking robustness is not equivalent to clinical correctness.'}}
assert len(conf)==5 and (checks.identity_jaccard==1).all() and (decomp.max_reconstruction_error<1e-8).all()
summary={'dataset_total_rows':len(raw),'hospital_years':len(panel),'balanced_hospitals_all_years':len(common_hospitals),
         'yearly_reporting_windows':conf.set_index('year').reported_window_count.to_dict(),
         'negative_control_expected_direction_fraction':float((checks.actual_policy_jaccard>checks.shuffled_id_placebo_jaccard).mean()),
         'figures_png':len(list(FIG.glob('*.png'))),'objective_audit':audits,
         'ethics_note':'Aggregated CMS provider-level public data; no patient records; no claim of superior real clinical outcomes.'}
(V4/'metrics'/'objective_audit.json').write_text(json.dumps(summary,indent=2,default=str))
lines=['# ChronoQuality V4 — exploratory evidence audit','',f'- Records: {len(raw):,}',f'- Hospital-years: {len(panel):,}',f'- V4 figures: {summary["figures_png"]} (PNG/SVG/PDF each)','',
 '**Prior results V1–V3 informed the pivot: V4 is NOT an untouched prospective confirmation.**','',
 '## Objective assessment']
for key,a in audits.items():lines.extend([f'### {key}: {a["status"]}',f'- Evidence: {a["evidence"]}',f'- Limitations: {a["limitations"]}',''])
lines += ['## Alternative claims to consider','- If year/policy rank changes are substantial: focus on **measure-policy reporting sensitivity** rather than superior hospital-quality prediction.','- If robust top-k membership is low: focus on **decision-relevant leaderboard instability** under defensible scoring choices.','- If policy effects are negligible: report stability and change the claimed contribution to a reproducible **benchmarking audit protocol**, subject to prior-art evaluation.','',
 '## Required before strong IJMI submission','- Independent semantic review of comparable measures; stronger benchmarking method baselines; ideally another data source or held-out future release.','- The dataset contains synchronized windows within releases; do not claim real within-release temporal mismatch detection.']
(V4/'FINAL_V4_EVIDENCE.md').write_text('\n'.join(lines))
manifest=[]
for f in sorted(ROOT.rglob('*')):
 if f.is_file():manifest.append({'relative_path':str(f.relative_to(ROOT)),'bytes':f.stat().st_size})
(V4/'qc'/'output_manifest.json').write_text(json.dumps(manifest,indent=2))
OUT=Path('/kaggle/working/ChronoQuality_RUN_004_Results.zip')
with zipfile.ZipFile(OUT,'w',zipfile.ZIP_DEFLATED,compresslevel=4,allowZip64=True) as z:
 for f in sorted(ROOT.rglob('*')):
  if f.is_file():z.write(f,arcname=str(f.relative_to(ROOT.parent)))
print('FINAL V4 ZIP:',OUT,'size MB',__import__('builtins').round(OUT.stat().st_size/1048576,2),'files',len(manifest))
print((V4/'FINAL_V4_EVIDENCE.md').read_text()[:2200])

## V7 original — V5 — CRAC: Certified Ranking and Attribution under Comparability Constraints / V5-A. Predeclared admissible scoring policies and exact finite enumeration

*All original code for this stage is preserved.*

In [ ]:
# Original V7 cell 71 (merged stage)
# V5 initialization — distinct workspace and traceable configuration
import itertools, gc, traceback
from scipy.stats import mannwhitneyu, spearmanr
V5=ROOT/'v5'; V5FIG=V5/'figures'; V5TAB=V5/'tables'; V5QC=V5/'qc'; V5BOOT=V5/'bootstrap'
for d in (V5,V5FIG,V5TAB,V5QC,V5BOOT):d.mkdir(parents=True,exist_ok=True)
CRAC_CONFIG={'name':'CRAC_finite_policy_certification','seed':42,
 'top_k':[10,25,50,100,250], 'max_hospitals_demo':None,
 'min_usable_items':3,'bootstrap_reps':400,'min_subgroup_n':35,
 'policy_set':'4 aggregation choices on fixed common concepts + leave-one-out fixed mean + all-compatible-available mean (if independent)',
 'reference_policy':'fixed_mean', 'policy_costs':{'aggregation':1,'catalog':2,'drop_one':1},
 'years':[2016,2017,2018,2019,2020], 'analysis_label':'EXPLORATORY'}
(V5QC/'config.json').write_text(json.dumps(CRAC_CONFIG,indent=2))
print('V5 output',V5)


# Original V7 cell 73 (merged stage)
def crac_trimmed_row(x):
    a=np.asarray(x,dtype=float);a=a[np.isfinite(a)]
    if len(a)<3:return np.nan
    a=np.sort(a)
    return float(np.mean(a[1:-1])) if len(a)>=5 else float(a.mean())

def crac_policy_matrix(year):
    X=panel.loc[year].copy()
    common=sorted(LOCKED_CONCEPTS)
    X=X.reindex(columns=common)
    policies={'fixed_mean':X.mean(axis=1), 'fixed_median':X.median(axis=1),
       'fixed_trimmed':pd.Series([crac_trimmed_row(v) for v in X.to_numpy()],index=X.index)}
    # Include all compatible positive-response measures for this particular release.
    # This is an observational reporting-policy alternative, NOT external clinical validation.
    a=pos[(pos.Year==year)&pos.answer_num.between(0,100)].copy()
    a=a.drop_duplicates(['facility','concept'],keep=False)
    full=a.pivot(index='facility',columns='concept',values='answer_num').reindex(X.index)
    policies['all_available_mean']=full.mean(axis=1)
    for column in common:
        if len(common)>CRAC_CONFIG['min_usable_items']:
            policies['drop:'+str(column)[:45]]=X.drop(columns=column).mean(axis=1)
    P=pd.DataFrame(policies)
    eligible=(X.notna().sum(axis=1)>=CRAC_CONFIG['min_usable_items']) & P.notna().all(axis=1)
    P=P.loc[eligible].sort_index()
    # Exact duplicate policies yield no novel rank orders; retain one canonical representative.
    keep=[]; duplicate_of={}
    for col in P.columns:
        found=next((prev for prev in keep if np.allclose(P[col],P[prev],atol=1e-10,rtol=0,equal_nan=True)),None)
        if found is None:keep.append(col)
        else:duplicate_of[col]=found
    return P[keep],X.loc[P.index],duplicate_of

def crac_topk(score,k):
    # Stable tie-break; not a claim that tied hospitals are clinically separable.
    s=pd.Series(score).dropna().sort_index()
    return set(s.sort_values(ascending=False,kind='mergesort').head(k).index)

def crac_ranks(P):
    return P.rank(ascending=False,method='min')

policy_by_year={}; policy_meta=[]
for y in CRAC_CONFIG['years']:
    P,X,dupes=crac_policy_matrix(y);policy_by_year[y]=P
    for c in P.columns:policy_meta.append({'year':y,'policy':c,'hospital_count':len(P),'deduplicated_aliases':str([x for x,v in dupes.items() if v==c])})
    assert P.index.is_unique and P.notna().all().all()
    print(y,'eligible hospitals',len(P),'distinct policies',len(P.columns),'dupes',len(dupes))
pd.DataFrame(policy_meta).to_csv(V5TAB/'policy_registry.csv',index=False)


# Original V7 cell 75 (merged stage)
def crac_cost(policy):
    if policy=='fixed_mean':return 0
    if policy.startswith('drop:'):return CRAC_CONFIG['policy_costs']['drop_one']
    if policy=='all_available_mean':return CRAC_CONFIG['policy_costs']['catalog']
    return CRAC_CONFIG['policy_costs']['aggregation']

cert_rows=[]; flip_rows=[]; cert_summary=[]
for year,P in policy_by_year.items():
    ranks=crac_ranks(P)
    for k in CRAC_CONFIG['top_k']:
        if len(P)<=k:continue
        mem=pd.DataFrame({p:pd.Series(P.index.isin(crac_topk(P[p],k)),index=P.index,dtype=bool) for p in P.columns})
        membership_n=mem.sum(axis=1)
        status=np.where(membership_n==len(P.columns),'certified_in',np.where(membership_n==0,'certified_out','sensitive'))
        cert_summary.append({'year':year,'k':k,'n':len(P),'policies':len(P.columns),
         'certified_in':int((status=='certified_in').sum()),'certified_out':int((status=='certified_out').sum()),
         'sensitive':int((status=='sensitive').sum()), 'certificate_rate':float((status!='sensitive').mean()),
         'robust_topk_fraction':float((status=='certified_in').sum()/k)})
        ref=mem['fixed_mean'];
        for fac in P.index:
            changes=[p for p in P.columns if bool(mem.at[fac,p])!=bool(ref.at[fac])]
            ordered=sorted(changes,key=lambda p:(crac_cost(p),p))
            flip=ordered[0] if ordered else None
            cert_rows.append({'year':year,'facility':fac,'k':k,'status':status[P.index.get_loc(fac)],
                 'reference_topk':bool(ref.at[fac]),'policies_in_topk':int(membership_n.at[fac]),
                 'total_policies':len(P.columns),'membership_rate':float(membership_n.at[fac]/len(P.columns)),
                 'rank_min':float(ranks.loc[fac].min()),'rank_max':float(ranks.loc[fac].max())})
            if flip is not None:
                flip_rows.append({'year':year,'facility':fac,'k':k,'reference_membership':bool(ref.at[fac]),
                   'minimal_edit_cost':crac_cost(flip),'flip_policy':flip,'n_flip_policies':len(changes),
                   'reference_rank':float(ranks.at[fac,'fixed_mean']),'flip_rank':float(ranks.at[fac,flip])})
cert=pd.DataFrame(cert_rows); flips=pd.DataFrame(flip_rows); certsum=pd.DataFrame(cert_summary)
for name,obj in [('certificates',cert),('minimal_policy_flips',flips),('certificate_summary',certsum)]:obj.to_csv(V5TAB/(name+'.csv'),index=False)
print(certsum.to_string(index=False,max_rows=25))


# Original V7 cell 77 (merged stage)
check_rows=[]; test_rng=np.random.default_rng(505)
for year,P in policy_by_year.items():
    ids=test_rng.choice(P.index.to_numpy(),size=min(50,len(P)),replace=False)
    k=25
    top_lists={p:crac_topk(P[p],k) for p in P.columns}
    subset=cert[(cert.year==year)&(cert.k==k)].set_index('facility')
    for h in ids:
        brute=[h in top_lists[p] for p in P.columns]
        expected='certified_in' if all(brute) else ('certified_out' if not any(brute) else 'sensitive')
        assert subset.at[h,'status']==expected
    # Affine transformation of ALL numeric score values preserves each policy ordering.
    Pa=P*1.25+7
    for policy in P.columns:
        assert crac_topk(P[policy],25)==crac_topk(Pa[policy],25)
    # Stable row-order invariance and deterministic tie-breaking by hospital ID.
    perm=P.sample(frac=1,random_state=year)
    for policy in P.columns:
        assert crac_topk(P[policy],25)==crac_topk(perm[policy],25)
    check_rows.append({'year':year,'random_exact_check_hospitals':len(ids),'exact_matches':len(ids),
      'affine_invariance':True,'row_order_invariance':True})
checks5=pd.DataFrame(check_rows);checks5.to_csv(V5QC/'exactness_invariance_checks.csv',index=False)
print(checks5.to_string(index=False))


## V7 original — V5-D. Subgroup instability with peer-relative comparison / V5-E. Paired uncertainty and group heterogeneity

*All original code for this stage is preserved.*

In [ ]:
# Original V7 cell 79 (merged stage)
meta5=meta.copy().reset_index()
meta5['facility']=meta5['facility'].astype(str)
meta5['survey_band']=pd.cut(meta5['survey_n'],[-np.inf,99,299,999,np.inf],labels=['<100 measured','100-299','300-999','1000+']).astype(object)
meta5.loc[meta5.survey_n.isna(),'survey_band']='censored_or_unavailable'
meta5['response_band']=pd.cut(meta5['response_pct'],[-np.inf,19.999,29.999,np.inf],labels=['<20%','20-<30%','30%+']).astype(object)
meta5.loc[meta5.response_pct.isna(),'response_band']='unavailable'
subgroup_rows=[]; peer_cert_rows=[]
for year,P in policy_by_year.items():
    mm=meta5[meta5.Year==year].drop_duplicates('facility').set_index('facility').reindex(P.index)
    c=cert[(cert.year==year)&(cert.k==100)].set_index('facility')
    base=mm.join(c[['status']],how='inner')
    for variable in ['hospital_type','survey_band','response_band','state']:
        for group,g in base.groupby(variable,dropna=False):
            if len(g)<CRAC_CONFIG['min_subgroup_n']:continue
            subgroup_rows.append({'year':year,'variable':variable,'group':str(group),'n':len(g),
              'global_top100_sensitive_rate':float((g.status=='sensitive').mean()),
              'global_top100_certified_in_rate':float((g.status=='certified_in').mean())})
    # Peer-relative top-k (10% of peer hospitals), not fixed national top 100.
    for ht,group in mm.groupby('hospital_type'):
        group_ids=group.index
        if len(group_ids)<CRAC_CONFIG['min_subgroup_n']:continue
        k=max(10,int(__import__('builtins').round(0.1*len(group_ids))))
        kk=min(k,len(group_ids)-1)
        PP=P.loc[group_ids]
        memberships=np.column_stack([PP.index.isin(crac_topk(PP[p],kk)) for p in PP.columns])
        sensitivity=(memberships.any(axis=1)&~memberships.all(axis=1))
        for i,h in enumerate(PP.index):
            peer_cert_rows.append({'year':year,'facility':h,'hospital_type':str(ht),'peer_n':len(PP),
                'peer_k':kk,'peer_sensitive':bool(sensitivity[i]),
                'peer_in_all':bool(memberships[i].all()),'peer_in_any':bool(memberships[i].any())})
subgroups=pd.DataFrame(subgroup_rows);peer_cert=pd.DataFrame(peer_cert_rows)
subgroups.to_csv(V5TAB/'subgroups_global_top100.csv',index=False)
peer_cert.to_csv(V5TAB/'peer_relative_certificates.csv',index=False)
peer_summary=peer_cert.groupby(['year','hospital_type']).agg(n=('facility','size'),
 sensitive_rate=('peer_sensitive','mean'),always_top=('peer_in_all','sum'),sometimes_top=('peer_in_any','sum')).reset_index()
peer_summary.to_csv(V5TAB/'peer_type_summary.csv',index=False)
print(peer_summary.to_string(index=False))


# Original V7 cell 81 (merged stage)
rng5=np.random.default_rng(5505)
boot_rows=[]; B=CRAC_CONFIG['bootstrap_reps']
for year,g in peer_cert.groupby('year'):
    groups={str(name):a.peer_sensitive.to_numpy(dtype=float) for name,a in g.groupby('hospital_type') if len(a)>=CRAC_CONFIG['min_subgroup_n']}
    if len(groups)<2:continue
    names=sorted(groups,key=lambda n:len(groups[n]),reverse=True)[:2]
    A,Bb=groups[names[0]],groups[names[1]]
    obs=float(A.mean()-Bb.mean())
    reps=np.empty(B)
    for b in range(B):
        reps[b]=A[rng5.integers(len(A),size=len(A))].mean()-Bb[rng5.integers(len(Bb),size=len(Bb))].mean()
    boot_rows.append({'year':year,'group_a':names[0],'group_b':names[1],
       'n_a':len(A),'n_b':len(Bb),'observed_difference':obs,
       'ci_low':float(np.quantile(reps,.025)),'ci_high':float(np.quantile(reps,.975)),
       'replicates':B,'analysis':'descriptive_not_causal'})
subboot=pd.DataFrame(boot_rows);subboot.to_csv(V5BOOT/'peer_group_cluster_bootstrap.csv',index=False)
print(subboot.to_string(index=False))


# Original V7 cell 83 (merged stage)
frontier=[]
for year,P in policy_by_year.items():
    regimes={'aggregations':[c for c in P if c in ['fixed_mean','fixed_median','fixed_trimmed']],
             'aggregations_plus_LOO':[c for c in P if c in ['fixed_mean','fixed_median','fixed_trimmed'] or c.startswith('drop:')],
             'complete_admissible':list(P.columns)}
    previous=None
    for regime,cols in regimes.items():
        if not cols:continue
        started=time.perf_counter()
        k=min(100,len(P)-1)
        mm=np.column_stack([P.index.isin(crac_topk(P[p],k)) for p in cols])
        stable=int(mm.all(axis=1).sum())
        sensitive=int((mm.any(axis=1)&~mm.all(axis=1)).sum())
        frontier.append({'year':year,'regime':regime,'policy_count':len(cols),
          'certified_in':stable,'sensitive':sensitive,'robust_topk_fraction':stable/k,
          'runtime_seconds':time.perf_counter()-started})
        if previous is not None:assert stable<=previous,'Adding permissible policies cannot increase stable-in count.'
        previous=stable
frontier=pd.DataFrame(frontier);frontier.to_csv(V5TAB/'baseline_policy_frontier.csv',index=False)
print(frontier.to_string(index=False))


# Original V7 cell 85 (merged stage)
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size':10,'axes.spines.top':False,'axes.spines.right':False,'savefig.bbox':'tight'})
def save_v5(name):
    plt.savefig(V5FIG/(name+'.png'),dpi=300,bbox_inches='tight')
    plt.savefig(V5FIG/(name+'.pdf'),bbox_inches='tight')
    plt.savefig(V5FIG/(name+'.svg'),bbox_inches='tight')
    plt.close()

# 1: Full certification landscape over year/k
p=certsum.pivot(index='k',columns='year',values='robust_topk_fraction')
fig,ax=plt.subplots(figsize=(8,4.6));im=ax.imshow(p.values,aspect='auto',vmin=0,vmax=1,cmap='viridis')
ax.set_xticks(np.arange(len(p.columns)),p.columns);ax.set_yticks(np.arange(len(p.index)),p.index)
ax.set(xlabel='Release year',ylabel='Top-k threshold',title='Exact finite-policy top-k certification')
fig.colorbar(im,ax=ax,label='Certified-in / k');save_v5('F01_certification_heatmap')

# 2: policy-set frontier
fig,ax=plt.subplots(figsize=(8,4.5))
for regime,g in frontier.groupby('regime'):
    ax.plot(g.year,g.robust_topk_fraction,marker='o',label=regime)
ax.set(xlabel='Release year',ylabel='Certified top-100 fraction',title='Certification versus admissible policy breadth',ylim=(0,1))
ax.legend(frameon=False,fontsize=8);save_v5('F02_policy_frontier')

# 3: minimal policy edit-cost histogram
fig,ax=plt.subplots(figsize=(7,4))
if not flips.empty:
    flips.minimal_edit_cost.value_counts().sort_index().plot(kind='bar',ax=ax)
ax.set(xlabel='Minimum policy-edit cost',ylabel='Hospital-year-k decision flips',title='Minimal admissible decision reversals');save_v5('F03_minimal_flip_cost')

# 4: year-by-year sensitive population for k=100
fig,ax=plt.subplots(figsize=(8,4))
g=certsum[certsum.k==100].sort_values('year')
ax.bar(g.year,100*g.sensitive/g.n)
ax.set(xlabel='Release year',ylabel='Policy-sensitive hospitals (%)',title='Fraction of hospital rankings sensitive to policy set');save_v5('F04_sensitive_share')

# 5: group profile, across years
fig,ax=plt.subplots(figsize=(8,4.4))
for group,g in peer_summary.groupby('hospital_type'):
    ax.plot(g.year,g.sensitive_rate,marker='o',label=str(group))
ax.set(xlabel='Release year',ylabel='Peer-relative sensitivity',ylim=(0,1),title='Within-type policy sensitivity');ax.legend(frameon=False,fontsize=8);save_v5('F05_hospital_type_peer_sensitivity')

# 6: subgroup differences with bootstrap intervals
fig,ax=plt.subplots(figsize=(8,4.5))
if len(subboot):
    y=np.arange(len(subboot)); vals=subboot.observed_difference.to_numpy()
    ax.errorbar(vals,y,xerr=np.array([vals-subboot.ci_low.to_numpy(),subboot.ci_high.to_numpy()-vals]),fmt='o',capsize=3)
    ax.set_yticks(y,subboot.year.astype(str)+' : '+subboot.group_a+' vs '+subboot.group_b)
ax.axvline(0,color='gray',linestyle='--',linewidth=1)
ax.set(xlabel='Between-group difference in sensitivity rate',title='Peer-relative subgroup differences (hospital bootstrap)');save_v5('F06_peer_subgroup_forest')

# 7: rank interval width distribution by year
fig,ax=plt.subplots(figsize=(8,4))
for year in CRAC_CONFIG['years']:
    sub=cert[(cert.year==year)&(cert.k==100)]
    ax.plot(np.quantile(sub.rank_max-sub.rank_min,[.1,.25,.5,.75,.9]),label=str(year),marker='o')
ax.set_xticks(range(5),['10th','25th','Median','75th','90th'])
ax.set(xlabel='Hospital rank-interval quantile',ylabel='Rank range',title='Finite-policy rank uncertainty distribution');ax.legend(frameon=False);save_v5('F07_rank_interval_quantiles')

# 8: rank interval panel for most policy-sensitive hospitals
fig,ax=plt.subplots(figsize=(8,6))
t=cert[(cert.year==2020)&(cert.k==100)&(cert.status=='sensitive')].copy()
t['width']=t.rank_max-t.rank_min
t=t.nsmallest(20,'rank_min').sort_values('rank_min',ascending=False)
for i,row in enumerate(t.itertuples()):ax.plot([row.rank_min,row.rank_max],[i,i],marker='|',linewidth=2)
ax.set_yticks(range(len(t)),[str(x)[-5:] for x in t.facility]);ax.set(xlabel='Best to worst admissible rank',ylabel='Hospital ID suffix',title='Illustrative 2020 policy-sensitive rank certificates');save_v5('F08_example_rank_certificates')

# 9: peer top robustness across hospital classes
fig,ax=plt.subplots(figsize=(8,4))
for h,g in peer_summary.groupby('hospital_type'):
    ax.plot(g.year,g.always_top/g['n'],marker='o',label=str(h))
ax.set(xlabel='Release year',ylabel='Certified peer top membership / peer hospitals',title='Peer membership stability across policies');ax.legend(frameon=False,fontsize=8);save_v5('F09_peer_certified_membership')

# 10: cost vs stability scatter
fig,ax=plt.subplots(figsize=(7,4.5))
for regime,g in frontier.groupby('regime'):
    ax.scatter(g.policy_count,g.robust_topk_fraction,label=regime,alpha=.75)
ax.set(xlabel='Number of admissible policies',ylabel='Certified-in / top-100',ylim=(0,1),title='Certifiability and policy-set size');ax.legend(frameon=False,fontsize=8);save_v5('F10_policy_count_robustness')

# 11: exact method regression correctness
fig,ax=plt.subplots(figsize=(7,3.8))
ax.bar(checks5.year,checks5.exact_matches/checks5.random_exact_check_hospitals)
ax.set(xlabel='Release year',ylabel='Exact match rate',ylim=(0,1.05),title='Independent brute-force certificate validation');save_v5('F11_exactness_check')
print('V5 figures',len(list(V5FIG.glob('*.png'))))


# Original V7 cell 87 (merged stage)
summary5={'version':'5.0_CRAC','exploratory':True,'input_records':int(len(raw)),
 'hospital_years':int(len(panel)),'distinct_policy_counts':{str(y):int(len(p.columns)) for y,p in policy_by_year.items()},
 'validated_exactness':bool((checks5.exact_matches==checks5.random_exact_check_hospitals).all()),
 'peer_subgroup_rows':int(len(peer_cert)),'certificate_rows':int(len(cert)),
 'flip_explanations':int(len(flips)),'figures':int(len(list(V5FIG.glob('*.png')))),
 'limitations':['Finite admissible policy set only; no guarantee outside it',
 'Overall quality ratings are not independent clinical ground truth',
 'Hospital type differences descriptive, not causal or quality-equity conclusions',
 '2019/2020 already used in prior iterative analysis; not a newly untouched test set',
 'Not a clinical implementation or expert-validated measurement equivalence study']}
(V5QC/'objective_audit.json').write_text(json.dumps(summary5,indent=2))
report=['# ChronoQuality V5 CRAC — exploratory findings','',
 '## Research objectives / evidence','',
 '1. **Exact finite-policy certificates:** IMPLEMENTED; computationally checked against an independent brute-force evaluator. Not global clinical truth.',
 '2. **Minimum admissible policy edits:** IMPLEMENTED as least-cost single edit among enumerated alternatives. Multi-step continuous policies and causal claims are outside scope.',
 '3. **Subgroup robustness:** IMPLEMENTED within hospital-type peers and descriptively for other groups. Requires clinical contextualization and careful multiple-testing design.',
 '4. **Comparison with simpler sensitivity methods:** OBSERVATIONAL policy-set breadth frontier; cannot interpret higher certificate rate alone as superiority.',
 '5. **Temporal replication:** EXPLORATORY, since 2019–2020 already informed earlier versions.','',
 '## Key numerical outcomes',
 f'- Certificate records: {len(cert):,}; minimal flip explanations: {len(flips):,}.',
 f'- Exact checks: {int(checks5.exact_matches.sum())}/{int(checks5.random_exact_check_hospitals.sum())}.',
 f'- Peer-relative subgroup rows: {len(peer_cert):,}.',
 f'- Figures: {summary5["figures"]} PNG/SVG/PDF triples.','',
 '## Certificate summary',certsum.to_markdown(index=False),'',
 '## Hospital-type subgroup summary',peer_summary.to_markdown(index=False),'',
 '## Main limitations','\n'.join('- '+x for x in summary5['limitations']),'',
 '## Scope of novelty',
 'CRAC is proposed as auditable finite-policy ranking certificates, minimal admissible flip witnesses and peer-relative stability auditing. Novelty versus prior robust ranking methods remains unverified and must be reviewed before IJMI submission.']
(V5/'FINAL_V5_EVIDENCE.md').write_text('\n'.join(report))
files5=[]
for file in sorted(ROOT.rglob('*')):
 if file.is_file() and not file.name.endswith('.zip'):
  files5.append({'relative_path':str(file.relative_to(ROOT)),'bytes':file.stat().st_size})
(V5QC/'output_inventory.json').write_text(json.dumps(files5,indent=2))
FINAL5=Path('/kaggle/working/ChronoQuality_RUN_005_Results.zip')
with zipfile.ZipFile(FINAL5,'w',zipfile.ZIP_DEFLATED,compresslevel=4,allowZip64=True) as z:
 for file in sorted(ROOT.rglob('*')):
  if file.is_file() and not file.name.endswith('.zip'):
   z.write(file,arcname=str(file.relative_to(ROOT.parent)))
print('Completed:',FINAL5,'MB:',__import__('builtins').round(FINAL5.stat().st_size/1048576,2),'outputs:',len(files5))


## V7 original — V6 — IJMI-focused CRAC methodological validation (retrospective, exploratory) / V6.2 — Independent exact rank-membership certificates

*All original code for this stage is preserved.*

In [ ]:
# Original V7 cell 89 (merged stage)
# V6.1 configuration and provenance
import os, json, time, gc, sys, platform, hashlib, zipfile
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import spearmanr, chi2_contingency
V6=ROOT/'v6'; T6=V6/'tables'; F6=V6/'figures'; Q6=V6/'qc'; M6=V6/'metrics'
for d in [T6,F6,Q6,M6]:d.mkdir(parents=True,exist_ok=True)
C6={'seed':4206,'k_values':[10,25,50,100,250],'benchmark_sizes':[100,300,750,1500],
    'subgroup_model':'binomial_GLM_hospital_clustered_standard_errors',
    'primary_outcome':'peer_sensitive','primary_exposure':'hospital_type',
    'multiple_testing':'BH_FDR_for_exploratory_subgroup_p_values',
    'validation':'retrospective_exploratory_no_unseen_external_test',
    'policy_scope':'finite_registered_policy_set','figure_dpi':300}
(Q6/'locked_protocol.json').write_text(json.dumps(C6,indent=2))
print('V6 setup:',V6)


# Original V7 cell 91 (merged stage)
# Complete exhaustive verification rather than sampled exactness tests
exact_rows=[]; miss=[]; verify=[]
for y,P in policy_by_year.items():
    sorted_members={p:{} for p in P.columns}
    for p in P.columns:
        order=P[p].sort_index().sort_values(ascending=False,kind='mergesort').index.to_numpy()
        for k in C6['k_values']:
            sorted_members[p][k]=set(order[:min(k,len(order))])
    for k in C6['k_values']:
        S=cert.loc[(cert.year==y)&(cert.k==k)].set_index('facility')
        assert len(S)==len(P)
        member=np.column_stack([P.index.isin(sorted_members[p][k]) for p in P.columns])
        truth=np.where(member.all(axis=1),'certified_in',np.where(~member.any(axis=1),'certified_out','sensitive'))
        predicted=S.reindex(P.index)['status'].to_numpy()
        nerr=int(np.sum(truth!=predicted))
        exact_rows.append({'year':y,'k':k,'hospital_count':len(P),'policies':len(P.columns),
                'evaluated_certificates':len(P),'mismatches':nerr,'exact_accuracy':1-nerr/len(P),
                'robust_topk_fraction':int(member.all(axis=1).sum())/k,
                'sensitive_count':int(np.sum(truth=='sensitive'))})
        if nerr:miss.extend([{'year':y,'k':k,'hospital':str(h),'expected':str(t),'actual':str(a)} for h,t,a in zip(P.index,truth,predicted) if t!=a])
exact6=pd.DataFrame(exact_rows); exact6.to_csv(T6/'exact_certificate_verification.csv',index=False)
pd.DataFrame(miss,columns=['year','k','hospital','expected','actual']).to_csv(Q6/'certificate_mismatches.csv',index=False)
print('Certificate checks:',int(exact6.evaluated_certificates.sum()),'mismatches:',int(exact6.mismatches.sum()))
assert exact6.mismatches.sum()==0,'Certificate mismatch; stop before interpreting outputs'


# Original V7 cell 93 (merged stage)
flipcheck=[]; bad=[]
for y,P in policy_by_year.items():
    f=flips.loc[flips.year==y]
    if f.empty:continue
    for k,ff in f.groupby('k'):
        ranks={p:crac_topk(P[p],int(k)) for p in P.columns}
        for row in ff.itertuples(index=False):
            h=row.facility; ref=h in ranks['fixed_mean']; target=h in ranks[row.flip_policy]
            candidates=[(crac_cost(p),p) for p in P.columns if (h in ranks[p])!=ref]
            mincost=min([c for c,p in candidates],default=np.inf)
            valid=bool(ref!=target and int(row.minimal_edit_cost)==mincost and row.reference_membership==ref)
            flipcheck.append({'year':y,'k':k,'facility':h,'reference_membership':ref,
               'flip_policy':row.flip_policy,'flip_membership':target,
               'recorded_cost':int(row.minimal_edit_cost),'recomputed_min_cost':mincost,'valid':valid})
flip6=pd.DataFrame(flipcheck);flip6.to_csv(T6/'all_flip_witness_checks.csv',index=False)
print('Flip witnesses checked:',len(flip6),'invalid:',int((~flip6.valid).sum()))
assert flip6.valid.all(),'Invalid policy-flip witnesses detected'


# Original V7 cell 95 (merged stage)
# Both implementations evaluate the same finite policy set and same deterministic tie-break.
bench=[]
for y in [2016,2019,2020]:
 P=policy_by_year[y]
 for N in C6['benchmark_sizes']:
  if N>len(P):continue
  S=P.iloc[:N].copy();k=min(25,N-1)
  t=time.perf_counter()
  orders={p:list(S[p].sort_index().sort_values(ascending=False,kind='mergesort').index[:k]) for p in S}
  slow={h:('certified_in' if all(h in orders[p] for p in S) else 'certified_out' if all(h not in orders[p] for p in S) else 'sensitive') for h in S.index}
  slow_seconds=time.perf_counter()-t
  t=time.perf_counter()
  membership=np.column_stack([S.index.isin(crac_topk(S[p],k)) for p in S])
  quick=np.where(membership.all(axis=1),'certified_in',np.where(~membership.any(axis=1),'certified_out','sensitive'))
  fast_seconds=time.perf_counter()-t
  assert list(slow.values())==list(quick)
  bench.append({'year':y,'hospitals':N,'policies':len(S.columns),'k':k,'literal_seconds':slow_seconds,
                'vectorized_seconds':fast_seconds,'speedup':slow_seconds/max(fast_seconds,1e-9),'exact_agreement':1.0})
bench6=pd.DataFrame(bench);bench6.to_csv(T6/'algorithm_runtime_comparison.csv',index=False)
print(bench6.to_string(index=False))


# Original V7 cell 97 (merged stage)
front=[]; stability=[]
for y,P in policy_by_year.items():
 regs={'reference_only':['fixed_mean'],
  'aggregations':[p for p in P if p in ('fixed_mean','fixed_median','fixed_trimmed')],
  'plus_indicator_drop':[p for p in P if p in ('fixed_mean','fixed_median','fixed_trimmed') or p.startswith('drop:')],
  'complete':list(P.columns)}
 for k in C6['k_values']:
  if len(P)<=k:continue
  for regime,pols in regs.items():
   mem=np.column_stack([P.index.isin(crac_topk(P[p],k)) for p in pols])
   front.append({'year':y,'k':k,'regime':regime,'policies':len(pols),'robust_in':int(mem.all(axis=1).sum()),
       'unstable':int((mem.any(axis=1)&~mem.all(axis=1)).sum()),'robust_fraction':float(mem.all(axis=1).sum()/k)})
 f=pd.DataFrame([x for x in front if x['year']==y and x['k']==100])
 assert f.robust_in.iloc[-1]<=f.robust_in.iloc[0]
front6=pd.DataFrame(front);front6.to_csv(T6/'policy_set_frontier_v6.csv',index=False)


## V7 original — V6.6 — Adjusted repeated-hospital subgroup analysis / V6.7 — Subgroup comparisons by ownership, volume, response and hospital category

*All original code for this stage is preserved.*

In [ ]:
# Original V7 cell 99 (merged stage)
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
m6=meta5[['Year','facility','hospital_type','survey_band','response_band','state']].drop_duplicates(['Year','facility'])
# Peer certificates use 'year', while metadata uses 'Year'. Join on explicit, aligned keys.
assert {'year','facility','hospital_type'}.issubset(peer_cert.columns)
assert not peer_cert.duplicated(['year','facility']).any(), 'Repeated peer certificate hospital-year'
assert not m6.duplicated(['Year','facility']).any(), 'Repeated metadata hospital-year'
a6=peer_cert.merge(m6.drop(columns=['hospital_type']),
    left_on=['year','facility'],right_on=['Year','facility'],
    how='left',validate='one_to_one',indicator=True)
assert (a6['_merge']=='both').all(), 'Unmatched peer hospital/year metadata'
a6=a6.drop(columns=['_merge','Year'])
a6['peer_sensitive_int']=a6['peer_sensitive'].astype(int)
a6=a6.rename(columns={'year':'release_year'})
for c in ['hospital_type','survey_band','response_band','state']:
 a6[c]=a6[c].fillna('unavailable').astype(str)
# Retain category labels, including censored_or_unavailable. State fixed effects control broad spatial heterogeneity.
formula='peer_sensitive_int ~ C(hospital_type) + C(survey_band) + C(response_band) + C(release_year) + C(state)'
a6.to_csv(T6/'subgroup_analysis_population.csv',index=False)
fit_error=None
try:
 mod=smf.glm(formula=formula,data=a6,family=sm.families.Binomial(),eval_env=-1)
 fit=mod.fit(cov_type='cluster',cov_kwds={'groups':a6['facility']},maxiter=150)
 ci=fit.conf_int()
 reg6=pd.DataFrame({'term':fit.params.index,'coefficient_log_odds':fit.params.values,
       'std_error_clustered':fit.bse.values,'p_value':fit.pvalues.values,
       'odds_ratio':np.exp(np.clip(fit.params.values,-20,20)),
       'or_lower_95':np.exp(np.clip(ci.iloc[:,0].values,-20,20)),
       'or_upper_95':np.exp(np.clip(ci.iloc[:,1].values,-20,20))})
 reg6['q_value_bh']=np.nan
 terms=reg6.term.str.startswith(('C(hospital_type)','C(survey_band)','C(response_band)'))
 usable=terms & reg6.p_value.notna()
 if usable.any():reg6.loc[usable,'q_value_bh']=multipletests(reg6.loc[usable,'p_value'],method='fdr_bh')[1]
 reg6.to_csv(T6/'adjusted_subgroup_clustered_glm.csv',index=False)
 fit_info={'n_observations':int(fit.nobs),'n_unique_hospitals':int(a6.facility.nunique()),
           'converged':bool(fit.converged),'formula':formula,'standard_errors':'hospital-clustered',
           'causal_claim':False}
except Exception as exc:
 fit_error=repr(exc);fit_info={'error':fit_error,'status':'not_established'}
 reg6=pd.DataFrame(columns=['term','odds_ratio','or_lower_95','or_upper_95','p_value','q_value_bh'])
 reg6.to_csv(T6/'adjusted_subgroup_clustered_glm.csv',index=False)
(Q6/'subgroup_model_diagnostics.json').write_text(json.dumps(fit_info,indent=2))
print('Adjusted subgroup model:',fit_info)
print(reg6.loc[reg6.term.str.contains('hospital_type',na=False)].to_string(index=False))


# Original V7 cell 101 (merged stage)
if 'Hospital Ownership' in raw.columns:
 ownership=raw[['Year','facility','Hospital Ownership']].drop_duplicates(['Year','facility'])
else:
 ownership=raw[['Year','facility']].drop_duplicates().copy()
 ownership['Hospital Ownership']='unavailable_not_in_source'
 assert len(ownership)>0
assert not ownership.duplicated(['Year','facility']).any(),'Conflicting hospital ownership metadata: manual resolution required'
sg=peer_cert.merge(meta5[['Year','facility','survey_band','response_band','state']].drop_duplicates(['Year','facility']),
    left_on=['year','facility'],right_on=['Year','facility'],how='left',validate='one_to_one',indicator=True)
assert (sg['_merge']=='both').all(), 'Subgroup metadata incomplete'
sg=sg.drop(columns=['_merge'])
sg=sg.merge(ownership,on=['Year','facility'],how='left',validate='one_to_one')
sg=sg.rename(columns={'Hospital Ownership':'ownership'})
group_rows=[]
for field in ['hospital_type','ownership','survey_band','response_band','state']:
 for (y,g),v in sg.groupby(['year',field],dropna=False):
  if len(v)<35:continue
  group_rows.append({'year':y,'grouping':field,'group':str(g),'n':len(v),
       'policy_sensitive_fraction':float(v.peer_sensitive.mean()),
       'robust_in_fraction':float(v.peer_in_all.mean())})
subgroups6=pd.DataFrame(group_rows);subgroups6.to_csv(T6/'all_subgroup_descriptive.csv',index=False)


# Original V7 cell 103 (merged stage)
rng=np.random.default_rng(C6['seed']); a=peer_cert.copy()
# Equal-weight hospital summary across observed years, then hospital-level bootstrap.
hospital_avg=a.groupby(['facility','hospital_type']).peer_sensitive.mean().reset_index()
types=hospital_avg.hospital_type.value_counts().index.tolist()
if len(types)>=2:
 types=types[:2]
 A=hospital_avg.loc[hospital_avg.hospital_type==types[0],'peer_sensitive'].to_numpy(float)
 B=hospital_avg.loc[hospital_avg.hospital_type==types[1],'peer_sensitive'].to_numpy(float)
 obs=float(A.mean()-B.mean());N=2000
 draws=np.empty(N)
 for i in range(N):
  draws[i]=float(A[rng.integers(0,len(A),len(A))].mean()-B[rng.integers(0,len(B),len(B))].mean())
 boot6={'group_a':types[0],'group_b':types[1],'n_hospitals_a':len(A),'n_hospitals_b':len(B),
   'difference_a_minus_b':obs,'ci_low':float(np.quantile(draws,.025)),'ci_high':float(np.quantile(draws,.975)),
   'bootstrap_reps':N,'cluster':'hospital','analysis':'exploratory_noncausal'}
else:boot6={'error':'Fewer than two available hospital types'}
(Q6/'longitudinal_bootstrap_summary.json').write_text(json.dumps(boot6,indent=2))
print(boot6)


# Original V7 cell 105 (merged stage)
plt.rcParams.update({'font.size':10,'axes.spines.top':False,'axes.spines.right':False,'savefig.bbox':'tight'})
def plot6(fig,name):
 for ext in ['png','svg','pdf']:
  fig.savefig(F6/(name+'.'+ext),dpi=300,facecolor='white')
 plt.close(fig)
# F1: policy robustness frontiers
fig,ax=plt.subplots(figsize=(9,5))
for y,g in front6.loc[(front6.k==100)].groupby('year'):
 ax.plot(np.arange(len(g)),g.robust_fraction.to_numpy(),marker='o',label=str(y))
ax.set_xticks(range(4),['Reference','Aggregation','+Leave-one-out','All registered']);ax.set_ylabel('Fraction robustly in top-100')
ax.set_title('Policy-set expansion and robust top-100 membership');ax.legend(title='Release',ncol=3)
plot6(fig,'F01_certification_frontier')
# F2: exactness and runtime
fig,ax=plt.subplots(figsize=(8,4.8))
for y,g in bench6.groupby('year'):
 ax.plot(g.hospitals,g.literal_seconds,marker='o',label=f'{y} literal')
 ax.plot(g.hospitals,g.vectorized_seconds,marker='s',linestyle='--',label=f'{y} vectorized')
ax.set_xlabel('Hospitals');ax.set_ylabel('Observed execution time (seconds)');ax.set_title('Exact certification: independent implementation benchmark');ax.legend(ncol=2,fontsize=8)
plot6(fig,'F02_exact_algorithm_runtime')
# F3: top-k certificate stability
fig,ax=plt.subplots(figsize=(8,4.8))
for y,g in exact6.groupby('year'):
 ax.plot(g.k,g.robust_topk_fraction,marker='o',label=str(y))
ax.set_xlabel('Top-k threshold');ax.set_ylabel('Robust fraction within top-k');ax.set_title('Decision robustness by leaderboard size');ax.legend(ncol=3)
plot6(fig,'F03_multik_certification')
# F4: flip cost distribution
fig,ax=plt.subplots(figsize=(7,4.5))
flip6.recorded_cost.value_counts().sort_index().plot(kind='bar',ax=ax)
ax.set_xlabel('Minimum registered single-policy flip cost');ax.set_ylabel('Verified flip witnesses');ax.set_title('Auditable decision-reversal witness costs')
plot6(fig,'F04_flip_witness_cost')
# F5: subgroup temporal panel
p=peer_summary.pivot(index='year',columns='hospital_type',values='sensitive_rate')
fig,ax=plt.subplots(figsize=(8,4.5));p.plot(ax=ax,marker='o');ax.set_ylabel('Peer-relative sensitivity fraction');ax.set_title('Ranking sensitivity across hospital types and releases');ax.legend(title='Hospital type')
plot6(fig,'F05_hospital_type_trend')
# F6: regression forest, when available
q=reg6.loc[reg6.term.str.startswith(('C(hospital_type)','C(survey_band)','C(response_band)'))].copy()
q=q.replace([np.inf,-np.inf],np.nan).dropna(subset=['odds_ratio','or_lower_95','or_upper_95'])
if len(q):
 q=q.sort_values('odds_ratio').tail(16)
 fig,ax=plt.subplots(figsize=(9,max(4,0.42*len(q)+1)))
 ys=np.arange(len(q));ax.errorbar(q.odds_ratio,ys,xerr=[q.odds_ratio-q.or_lower_95,q.or_upper_95-q.odds_ratio],fmt='o',capsize=3)
 ax.axvline(1,color='gray',linestyle='--');ax.set_yticks(ys,q.term)
 ax.set_xscale('log');ax.set_xlabel('Adjusted odds ratio (95% clustered CI)');ax.set_title('Exploratory adjusted subgroup associations')
 plot6(fig,'F06_adjusted_subgroup_forest')
# F7: robustness heatmap
heat=exact6.pivot(index='year',columns='k',values='robust_topk_fraction')
fig,ax=plt.subplots(figsize=(8,4.5));im=ax.imshow(heat.to_numpy(),vmin=0,vmax=1,aspect='auto',cmap='viridis')
ax.set_xticks(range(len(heat.columns)),list(heat.columns));ax.set_yticks(range(len(heat.index)),list(heat.index));ax.set_xlabel('k');ax.set_ylabel('Release')
for i in range(len(heat.index)):
 for j in range(len(heat.columns)):ax.text(j,i,f'{heat.iloc[i,j]:.2f}',color='white' if heat.iloc[i,j]<0.65 else 'black',ha='center',va='center',fontsize=9)
fig.colorbar(im,ax=ax,label='Robust top-k fraction');ax.set_title('Certificate stability across release and threshold');plot6(fig,'F07_certificate_heatmap')
# F8: hospital peer group descriptive top instability
q=subgroups6.loc[subgroups6.grouping=='hospital_type']
fig,ax=plt.subplots(figsize=(8,4.5))
for g,d in q.groupby('group'):
 ax.plot(d.year,d.policy_sensitive_fraction,marker='o',label=g)
ax.set_title('Peer-sensitive membership by hospital category');ax.set_xlabel('Release');ax.set_ylabel('Observed fraction');ax.legend()
plot6(fig,'F08_peer_subgroup_instability')
print('Saved V6 plots:',len(list(F6.glob('*.png'))))


# Original V7 cell 107 (merged stage)
# Gate rules are fixed and not selected to maximize favorable findings.
audit6={
 'source_files':5,'source_records':int(len(raw)),'hospital_year_records':int(len(panel)),
 'exact_certificates_checked':int(exact6.evaluated_certificates.sum()),
 'certificate_mismatch_count':int(exact6.mismatches.sum()),
 'flip_witnesses_checked':int(len(flip6)),'invalid_flip_witnesses':int((~flip6.valid).sum()),
 'finite_policy_correctness':bool(exact6.mismatches.sum()==0 and flip6.valid.all()),
 'adjusted_subgroup_model':fit_info,'hospital_level_bootstrap':boot6,
 'locked_pre_v6_development_vocabulary':True,
 'no_independent_test_data':True,'all_five_releases_seen_in_prior_research_iterations':True,
 'clinical_ground_truth_rank_available':False,
 'algorithmic_novelty_vs_prior_robust_ranking_proven':False,
 'clinically_deployed_decision_support_evaluated':False,
 'ready_for_IJMI_submission_automatically':False,
 'primary_claim':'Auditable policy-set robustness and validated registered flip witnesses, not true hospital-quality superiority',
 'exploratory':True}
(Q6/'FINAL_VALIDITY_AUDIT.json').write_text(json.dumps(audit6,indent=2,default=str))
report=['# ChronoQuality-CRAC V6: retrospective research evidence','',
 '## Scope','All findings remain retrospective and exploratory. Policy certificates are exact only over the registered finite policy set.','',
 '## Computational validation',f'- Hospital-policy certificate checks: {audit6["exact_certificates_checked"]:,}.',
 f'- Certificate disagreements: {audit6["certificate_mismatch_count"]:,}.',
 f'- Policy-flip witnesses verified: {audit6["flip_witnesses_checked"]:,}.',
 f'- Invalid witness records: {audit6["invalid_flip_witnesses"]:,}.','',
 '## Observational subgroup evidence',f'- Cluster-adjusted regression: {fit_info}.',f'- Repeated-hospital bootstrap: {boot6}.','',
 '## Claims supported','- Exact finite-policy enumeration, transparent instability summaries, and auditable flip witnesses when all gates pass.',
 '- Subgroup associations only when regression and interval diagnostics support them.','',
 '## Claims NOT established','- A universally minimal flip across continuous or multi-step policy spaces.',
 '- Correctness of the underlying hospital quality ranking or comparative clinical quality.',
 '- Independently confirmed clinical effects or causal impact of hospital type.',
 '- Methodological first-in-literature novelty or superiority over every established method.',
 '- Untouched 2019–2020 external validation; these periods already informed earlier analysis.',
 '- Acceptance by the International Journal of Medical Informatics.','',
 '## Next scientific decision','Use this package for reproducible computational evidence. Independent method comparisons, expert review of admissible measurement policies, and real independent validation are needed before submission-ready claims.']
(V6/'FINAL_V6_EVIDENCE.md').write_text('\n'.join(report))
print('\n'.join(report[:15]))

assert fit_info.get('converged',False), 'Adjusted subgroup model failed: inspect v6/qc/subgroup_model_diagnostics.json before manuscript export.'


## V7 original — V7 — Submission-focused correction and evidence freeze

*All original code for this stage is preserved.*

In [ ]:
# Original V7 cell 110 (merged stage)
# V7.1: Audit adjusted associations, prepare manuscript-ready quantitative results.
import numpy as np, pandas as pd, json
from pathlib import Path
V7=ROOT/'v7';V7TAB=V7/'tables';V7QC=V7/'qc'
for d in [V7,V7TAB,V7QC]: d.mkdir(parents=True,exist_ok=True)
assert fit_info.get('converged',False), 'Adjusted GLM did not converge: investigate before interpreting subgroup findings.'
assert not reg6.empty, 'Adjusted subgroup coefficients are missing.'
assert fit.nobs==len(a6), 'Regression unexpectedly dropped observations.'
assert len(a6)==a6[['release_year','facility']].drop_duplicates().shape[0]
assert bool(np.isfinite(reg6['coefficient_log_odds']).all()),'Non-finite estimates: model unusable.'
assert bool(np.isfinite(reg6['std_error_clustered']).all()),'Non-finite standard errors: model unusable.'
# Clear separation: unadjusted differences of prevalence are not adjusted regression ORs.
raw_groups=a6.groupby('hospital_type',as_index=False).agg(hospital_years=('facility','size'),
    unique_hospitals=('facility','nunique'),sensitive_share=('peer_sensitive_int','mean'))
raw_groups.to_csv(V7TAB/'unadjusted_hospital_type_rates.csv',index=False)
reg6.to_csv(V7TAB/'adjusted_subgroup_all_coefficients.csv',index=False)
ht=reg6[reg6.term.str.contains('hospital_type',case=False,na=False)].copy()
assert len(ht)>0,'No hospital-type parameter in adjusted model.'
ht.to_csv(V7TAB/'adjusted_hospital_type_effect.csv',index=False)
# Aggregate explicitly without treating 5 years as independent experiments.
summary7={'n_source_measure_rows':int(len(raw)),'n_hospital_years':int(len(panel)),
    'n_peer_hospital_years':int(len(a6)),'unique_peer_hospitals':int(a6.facility.nunique()),
    'regression_converged':bool(fit.converged),'regression_nobs':int(fit.nobs),
    'hospital_type_unadjusted':raw_groups.to_dict(orient='records'),
    'hospital_type_adjusted_coefficients':ht.to_dict(orient='records'),
    'possible_sign_reversal':bool(any((row['coefficient_log_odds']<0) for row in ht.to_dict(orient='records'))),
    'comparative_quality_truth_available':False,
    'prospective_or_independent_validation':False,
    'clinical_impact_evaluated':False,'journal_acceptance_guaranteed':False,
    'claim_scope':'Finite registered policy certificates; verified decision-flip witnesses; conditional observational associations.'}
(V7QC/'MANUSCRIPT_EVIDENCE_GATES.json').write_text(json.dumps(summary7,indent=2,default=str))
text7=['# ChronoQuality-CRAC V7 — Final evidence freeze','',
 '## Results that may be reported',
 '- Enumerated-policy ranking certificates and explicit sensitivity of top-k membership.',
 '- Verified decision-flip witnesses over admissible enumerated alternatives.',
 '- Hospital-type unadjusted rates and hospital-clustered adjusted GLM (report both, including any sign reversal).',
 '', '## Required contextual limits',
 '- Results are retrospective/exploratory; years previously used to refine this framework are not pristine external test sets.',
 '- No independent clinical-quality ranking, clinical outcome validation, or prospective deployment evaluation.',
 '- Exactness over a finite policy set is not proof of first-in-literature novel algorithmic capability.',
 '- Regression coefficients quantify conditional associations, not causal effects.',
 '- Policy choice, variable definition, patient-experience construct validity, and practical clinical informatics value require expert review.',
 '', '## Regression diagnostics',f'- Converged: {fit.converged}',f'- N observations: {int(fit.nobs)}',
 '','## Unadjusted hospital-type proportions',raw_groups.to_string(index=False),'',
 '## Adjusted hospital-type coefficients (clustered SE)',ht.to_string(index=False)]
(V7/'FINAL_V7_EVIDENCE.md').write_text('\n'.join(text7))
print('V7 adjusted subgroup model converged:',fit.converged)
print('Unadjusted proportions:');print(raw_groups.to_string(index=False))
print('Adjusted hospital type:');print(ht.to_string(index=False))


# Original V7 cell 111 (merged stage)
# Integrity manifest and final ZIP
files6=[]
for f in sorted(ROOT.rglob('*')):
 if f.is_file() and f.suffix!='.zip' and f!=Q6/'file_manifest.json':
  files6.append({'path':str(f.relative_to(ROOT)),'bytes':f.stat().st_size})
(Q6/'file_manifest.json').write_text(json.dumps(files6,indent=2))
FINAL6=Path('/kaggle/working/ChronoQuality_RUN_007_Results.zip')
with zipfile.ZipFile(FINAL6,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=4,allowZip64=True) as archive:
 for f in sorted(ROOT.rglob('*')):
  if f.is_file() and f.suffix!='.zip':archive.write(f,arcname=str(f.relative_to(ROOT.parent)))
with zipfile.ZipFile(FINAL6) as check:
 assert check.testzip() is None
print('Saved final V7 archive',FINAL6,'MB',__import__('builtins').round(FINAL6.stat().st_size/1048576,2),'files',len(files6))


# Reviewer-required correction module (standalone validation)

This branch reloads the same five source CSVs independently, reconstructs finite-policy ranking certificates and policy-flip witnesses, then carries out specification-sensitive regression, peer-group matching, denominator checks, and policy concept auditing. This independent reconstruction intentionally does **not** reuse V7's intermediate computed arrays. It writes outputs to a distinct directory to avoid overwriting V7. Reviewer resampling percentiles are *design sensitivity*, not independent clinical or patient-level confidence intervals.

## Reviewer validation — A. Official-data provenance and five-release source audit / B. Fixed vocabulary, semantic compatibility, hospital-year metadata

In [ ]:
# Reviewer source cell 2
import os, sys, json, time, platform, hashlib, zipfile, warnings, math, builtins
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf

SEED = 42
BALANCED_REPEATS = 500
BALANCED_FIXED_N = 600
HOSPITAL_BOOTSTRAP = 2000
DATA_HINT = ''  # e.g. '/kaggle/input/datasets/abrambeyer/us-hospital-customer-satisfaction-20162020'
EXPECTED_YEARS = (2016, 2017, 2018, 2019, 2020)
TOP_K = (10,25,50,100,250)
VALID_HOSPITAL_TYPES = ('Acute Care Hospitals', 'Critical Access Hospitals')
ROOT=Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path('/mnt/data')
OUT=ROOT/'ChronoQuality_CRAC_Standalone_Reviewer_Validation'
for sub in ('data','tables','figures','qc','notes'):(OUT/sub).mkdir(parents=True,exist_ok=True)

versions={'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__,
          'scipy':__import__('scipy').__version__,'matplotlib':matplotlib.__version__,'statsmodels':sm.__version__}
(OUT/'qc'/'dependency_versions.json').write_text(json.dumps(versions,indent=2))
print('All required packages available:', versions)

# Reviewer source cell 4
roots=[Path(DATA_HINT)] if DATA_HINT else [Path('/kaggle/input'),Path.cwd(),Path('/mnt/data')]
located={}
for yr in EXPECTED_YEARS:
    basename=f'cms_hospital_patient_satisfaction_{yr}.csv'
    hits=[]
    for root in roots:
        if root.is_file() and root.name==basename:hits.append(root)
        elif root.is_dir():hits.extend(root.rglob(basename))
    hits=sorted(set(hits),key=lambda f:(0 if '/kaggle/input/' in str(f) else 1,len(str(f))))
    if not hits:raise FileNotFoundError(f'Missing {basename}. Attach the five CMS-derived CSV files under Kaggle Add Input, or set DATA_HINT.')
    located[yr]=hits[0]
print('Five source files:', '\n'.join(f'{y}: {located[y]}' for y in EXPECTED_YEARS))

need=['Facility ID','Hospital Type','State','HCAHPS Measure ID','HCAHPS Question',
      'HCAHPS Answer Description','HCAHPS Answer Percent','Number of Completed Surveys',
      'Survey Response Rate Percent','Start Date','End Date','Year']
optional=['Hospital Ownership']
frames=[]; manifests=[]
for year,path in located.items():
    d=pd.read_csv(path,usecols=lambda col:col in set(need+optional),
                  dtype={'Facility ID':'string','HCAHPS Measure ID':'string'},low_memory=False)
    missing=set(need)-set(d.columns)
    if missing:raise ValueError(f'{path.name}: missing critical columns {sorted(missing)}')
    if 'Hospital Ownership' not in d:d['Hospital Ownership']=pd.NA
    years=set(pd.to_numeric(d['Year'],errors='coerce').dropna().astype(int).unique())
    if years!={year}:raise ValueError(f'Unexpected release year values in {path.name}: {years}')
    with open(path,'rb') as file_handle:
        sha=hashlib.file_digest(file_handle,'sha256').hexdigest() if hasattr(hashlib,'file_digest') else hashlib.sha256(path.read_bytes()).hexdigest()
    manifests.append({'file':path.name,'year':year,'rows':len(d),'sha256':sha,'bytes':path.stat().st_size})
    frames.append(d)
raw=pd.concat(frames,ignore_index=True)
for d in frames:del d
raw['facility']=raw['Facility ID'].str.strip().str.zfill(6)
raw['measure']=raw['HCAHPS Measure ID'].str.strip()
raw['question']=raw['HCAHPS Question'].astype('string').str.strip().str.lower().str.replace(r'\s+',' ',regex=True)
raw['answer_desc']=raw['HCAHPS Answer Description'].astype('string').str.strip().str.lower().str.replace(r'\s+',' ',regex=True)
raw['answer_num']=pd.to_numeric(raw['HCAHPS Answer Percent'],errors='coerce')
raw['survey_n']=pd.to_numeric(raw['Number of Completed Surveys'],errors='coerce')
raw['response_pct']=pd.to_numeric(raw['Survey Response Rate Percent'],errors='coerce')
raw['Start Date']=pd.to_datetime(raw['Start Date'],errors='coerce')
raw['End Date']=pd.to_datetime(raw['End Date'],errors='coerce')
raw['Year']=raw['Year'].astype(int)
release_audit=raw.groupby('Year',as_index=False).agg(measurement_rows=('facility','size'),
    unique_facilities=('facility','nunique'),measure_ids=('measure','nunique'),
    n_start_dates=('Start Date','nunique'),n_end_dates=('End Date','nunique'),
    first_start=('Start Date','min'),last_end=('End Date','max'))
release_audit.to_csv(OUT/'tables'/'input_release_audit.csv',index=False)
pd.DataFrame(manifests).to_csv(OUT/'qc'/'input_file_sha256.csv',index=False)
if len(raw)!=1653683:warnings.warn(f'Input has {len(raw):,} rows vs original research 1,653,683; do NOT copy prior numerical conclusions without rechecking.')
if not ((release_audit.n_start_dates==1)&(release_audit.n_end_dates==1)).all():
    warnings.warn('At least one release has multiple survey windows. Do not assume within-release alignment.')
print(release_audit.to_string(index=False));print('Input measurement rows:',len(raw))

# Reviewer source cell 6
pos=raw.loc[raw['measure'].str.endswith('_A_P',na=False)].copy()
pos['sem_key']=pos['question'].fillna('')+' || '+pos['answer_desc'].fillna('')
pos['concept']=pos['measure']+' || '+pos['sem_key']
concept_catalog=pos[['Year','measure','sem_key']].drop_duplicates().sort_values(['measure','Year'])
variation=pos.groupby('measure').agg(n_sem=('sem_key','nunique'),n_years=('Year','nunique'),rows=('facility','size')).reset_index()
concept_catalog.to_csv(OUT/'tables'/'historical_measure_catalog.csv',index=False)
variation.to_csv(OUT/'tables'/'historical_measure_variation.csv',index=False)
observed=pos[pos.Year.isin((2016,2017,2018))].groupby('concept').Year.nunique()
locked=sorted(observed.index[observed.eq(3)].tolist())
if not locked:
    locked=sorted(observed.index[observed.ge(2)].tolist())
if not locked:raise ValueError('No development-stable HCAHPS favorable-response concepts found.')
(OUT/'qc'/'locked_concepts.json').write_text(json.dumps(locked,indent=2))
valid=pos.loc[pos.concept.isin(locked)&pos.answer_num.between(0,100)&pos.facility.notna()].copy()
ambig=valid.duplicated(['Year','facility','concept'],keep=False)
ambiguous_count=int(ambig.sum())
valid=valid.loc[~ambig]
panel=valid.pivot(index=['Year','facility'],columns='concept',values='answer_num').sort_index()
if panel.index.duplicated().any():raise AssertionError('Duplicate hospital-year keys after pivot.')
meta=valid.groupby(['Year','facility'],sort=False).agg(state=('State','first'),hospital_type=('Hospital Type','first'),
    ownership=('Hospital Ownership','first'),survey_n=('survey_n','first'),response_pct=('response_pct','first'),
    start=('Start Date','first'),end=('End Date','first')).reindex(panel.index).reset_index()
meta['facility']=meta['facility'].astype(str).str.zfill(6)
meta['survey_band']=pd.cut(meta.survey_n,[-np.inf,99,299,999,np.inf],
    labels=['<100 measured','100-299','300-999','1000+']).astype(object)
meta.loc[meta.survey_n.isna(),'survey_band']='censored_or_unavailable'
meta['response_band']=pd.cut(meta.response_pct,[-np.inf,19.999,29.999,np.inf],
    labels=['<20%','20-<30%','30%+']).astype(object)
meta.loc[meta.response_pct.isna(),'response_band']='unavailable'
meta.to_csv(OUT/'data'/'hospital_year_metadata.csv',index=False)
panel.reset_index().to_csv(OUT/'data'/'fixed_six_concept_panel.csv',index=False)
print('Stable fixed concepts:',len(locked),'hospital years:',len(panel),'ambiguous repeated rows discarded:',ambiguous_count)
print([(int(y),int((panel.index.get_level_values('Year')==y).sum())) for y in EXPECTED_YEARS])
if len(locked)!=6:warnings.warn(f'Stable concept count {len(locked)} differs from manuscript count six; revise comparisons before publication.')

## Reviewer validation — C. Reconstruct scoring policies and independently verify exact top-k certificates / D. Compute peer-relative hospital rankings and verify the independently reconstructed status

In [ ]:
# Reviewer source cell 8
def trimmed_mean(values):
    arr=np.sort(np.asarray(values,dtype=float)[np.isfinite(values)])
    if len(arr)<3:return np.nan
    return float(arr[1:-1].mean()) if len(arr)>=5 else float(arr.mean())
def stable_topk_ids(series,k):
    s=pd.Series(series).dropna().sort_index()
    return set(s.sort_values(ascending=False,kind='mergesort').iloc[:k].index)
def top_mask(values,k):
    # Caller supplies sorted facility IDs; stable sort preserves ascending ID ties.
    arr=np.asarray(values,dtype=float)
    idx=np.argsort(-arr,kind='stable')[:k]
    m=np.zeros(len(arr),dtype=bool);m[idx]=True
    return m
def cost(policy):
    return 0 if policy=='fixed_mean' else (2 if policy=='all_available_mean' else 1)

policies_by_year={}; policy_rows=[]; cert_rows=[]; cert_summary=[]; flips=[]; independent=[]
for year in EXPECTED_YEARS:
    X=panel.loc[year].reindex(columns=locked).sort_index()
    source=pos[pos.Year.eq(year)&pos.answer_num.between(0,100)&pos.facility.notna()].copy()
    source=source.drop_duplicates(['facility','concept'],keep=False)
    all_mean=source.groupby('facility').answer_num.mean().reindex(X.index)
    variants={'fixed_mean':X.mean(axis=1),'fixed_median':X.median(axis=1),
              'fixed_trimmed':pd.Series([trimmed_mean(r) for r in X.to_numpy()],index=X.index),
              'all_available_mean':all_mean}
    for c in locked:
        if len(locked)>3:variants['drop:'+str(c)[:45]]=X.drop(columns=c).mean(axis=1)
    P=pd.DataFrame(variants,index=X.index)
    eligible=(X.notna().sum(axis=1)>=3)&P.notna().all(axis=1)
    P=P.loc[eligible].sort_index()
    retain=[]; aliases={}
    for p in P.columns:
        previous=next((old for old in retain if np.allclose(P[p],P[old],atol=1e-10,rtol=0,equal_nan=True)),None)
        if previous is None:retain.append(p)
        else: aliases[p]=previous
    P=P[retain]
    policies_by_year[year]=P
    for p in P.columns:policy_rows.append({'year':year,'policy':p,'cost':cost(p),'eligible_hospitals':len(P),
        'aliases':','.join(k for k,v in aliases.items() if v==p)})
    for k in TOP_K:
        if len(P)<=k:continue
        membership={p:np.asarray(P.index.isin(stable_topk_ids(P[p],k)),dtype=bool) for p in P.columns}
        matrix=np.column_stack(list(membership.values()))
        # Independent NumPy oracle for deterministic top-k membership.
        errors=0
        for p,j in zip(P.columns,range(P.shape[1])):
            oracle=top_mask(P[p].to_numpy(),k)
            errors+=int((oracle!=matrix[:,j]).sum())
        independent.append({'year':year,'k':k,'n_hospitals':len(P),'n_policies':P.shape[1],
            'membership_mismatches':errors})
        if errors:raise AssertionError(f'Independent top-k mismatch in year={year},k={k}: {errors}')
        n=matrix.sum(axis=1)
        status=np.where(n==matrix.shape[1],'certified_in',np.where(n==0,'certified_out','sensitive'))
        cert_summary.append({'year':year,'k':k,'eligible_hospitals':len(P),'policies':len(P.columns),
          'certified_in':int((status=='certified_in').sum()),'certified_out':int((status=='certified_out').sum()),
          'policy_sensitive':int((status=='sensitive').sum()),'robust_reference_topk_pct':100*(status=='certified_in').sum()/k})
        ref=membership['fixed_mean']
        rank=P.rank(axis=0,ascending=False,method='min')
        for i,fac in enumerate(P.index):
            cert_rows.append({'year':year,'facility':fac,'k':k,'status':status[i],
              'reference_topk':bool(ref[i]),'in_count':int(n[i]),'policies_count':P.shape[1]})
            possibilities=[p for j,p in enumerate(P.columns) if bool(matrix[i,j])!=bool(ref[i])]
            if possibilities:
                chosen=min(possibilities,key=lambda p:(cost(p),p))
                # independent direct re-sort of policy, sanity witness and registry cost
                changed=fac in stable_topk_ids(P[chosen],k)
                if changed==bool(ref[i]):raise AssertionError('Invalid policy-flip witness')
                if any(cost(p)<cost(chosen) for p in possibilities):raise AssertionError('Nonminimal registry cost')
                flips.append({'year':year,'facility':fac,'k':k,'reference_membership':bool(ref[i]),
                  'flip_policy':chosen,'minimum_registered_cost':cost(chosen),
                  'reference_rank_min_ties':float(rank.at[fac,'fixed_mean']),
                  'flip_rank_min_ties':float(rank.at[fac,chosen]),
                  'reversing_policy_count':len(possibilities)})
    del source
policy_registry=pd.DataFrame(policy_rows)
cert=pd.DataFrame(cert_rows)
certsum=pd.DataFrame(cert_summary)
flips=pd.DataFrame(flips)
for name,df in [('policy_registry',policy_registry),('certificates',cert),
                ('certificate_summary',certsum),('verified_policy_flips',flips),
                ('independent_membership_checks',pd.DataFrame(independent))]:
    df.to_csv(OUT/'tables'/(name+'.csv'),index=False)
print('Registry:',policy_registry.groupby('year').policy.count().to_dict())
print('Certificate evaluations:',len(cert),'mismatch:',sum(x['membership_mismatches'] for x in independent))
print('Verified reversal witnesses:',len(flips))
print(certsum.loc[certsum.k.eq(100)].to_string(index=False))

# Reviewer source cell 10
def sensitivity_for_policies(P):
    n=len(P)
    k=min(max(10,builtins.round(0.1*n)),n-1)
    arr=P.to_numpy(dtype=float)
    mat=np.column_stack([top_mask(arr[:,j],k) for j in range(arr.shape[1])])
    status=mat.any(axis=1)&(~mat.all(axis=1))
    return status,k,mat
peer_rows=[]; peer_check=[]
for year,P in policies_by_year.items():
    m=meta.loc[meta.Year.eq(year)].drop_duplicates('facility').set_index('facility').reindex(P.index)
    if m['hospital_type'].isna().any():raise ValueError(f'Missing peer hospital type in {year}')
    for grp,g in m.groupby('hospital_type'):
        if len(g)<35:continue
        PP=P.loc[sorted(g.index)]
        flags,k,mat=sensitivity_for_policies(PP)
        independently=np.column_stack([PP.index.isin(stable_topk_ids(PP[p],k)) for p in PP.columns])
        mismatch=int((mat!=independently).sum())
        if mismatch:raise AssertionError('Independent peer-group status mismatch')
        peer_check.append({'year':year,'group':str(grp),'n':len(PP),'peer_k':k,'membership_mismatch':mismatch})
        for i,fac in enumerate(PP.index):
            peer_rows.append({'release_year':year,'facility':fac,'hospital_type':str(grp),
                'peer_n':len(PP),'peer_k':k,'peer_sensitive_int':int(flags[i]),
                'peer_in_all':int(mat[i].all()),'peer_in_any':int(mat[i].any())})
peer=pd.DataFrame(peer_rows)
a=peer.loc[peer.hospital_type.isin(VALID_HOSPITAL_TYPES)].merge(meta[['Year','facility','state','ownership','survey_n','response_pct',
    'survey_band','response_band']],left_on=['release_year','facility'],right_on=['Year','facility'],
    validate='one_to_one',how='left',indicator=True)
if (a['_merge']!='both').any():raise AssertionError('Missing hospital-year metadata in peer analysis')
a=a.drop(columns=['Year','_merge'])
a['release_year']=a.release_year.astype(int)
for c in ['hospital_type','survey_band','response_band','state']:
    a[c]=a[c].fillna('unavailable').astype(str)
a.to_csv(OUT/'data'/'subgroup_analysis_population.csv',index=False)
pd.DataFrame(peer_check).to_csv(OUT/'qc'/'independent_peer_checks.csv',index=False)
print('Peer hospital-years:',len(a),'unique hospitals:',a.facility.nunique())
print(a.groupby('hospital_type').peer_sensitive_int.agg(['size','mean']).to_string())

## Reviewer validation — E. Reviewer correction — parsimonious adjusted repeated-hospital inference

In [ ]:
# Reviewer source cell 12
NORTHEAST=set('CT ME MA NH RI VT NJ NY PA'.split())
MIDWEST=set('IL IN MI OH WI IA KS MN MO NE ND SD'.split())
SOUTH=set('DE DC FL GA MD NC SC VA WV AL KY MS TN AR LA OK TX'.split())
WEST=set('AZ CO ID MT NV NM UT WY AK CA HI OR WA'.split())
def census_region(s):
    code=str(s).upper()
    return 'Northeast' if code in NORTHEAST else 'Midwest' if code in MIDWEST else 'South' if code in SOUTH else 'West' if code in WEST else 'Other/territory'
a['region']=a.state.map(census_region)
a['release_label']=a.release_year.astype(str)
FORMULAS={
 'A_year_hospital_type':'peer_sensitive_int ~ C(hospital_type) + C(release_label)',
 'B_plus_survey_context':'peer_sensitive_int ~ C(hospital_type) + C(release_label) + C(survey_band) + C(response_band)',
 'C_plus_broad_region':'peer_sensitive_int ~ C(hospital_type) + C(release_label) + C(survey_band) + C(response_band) + C(region)',
}
rows=[]
for col in ['hospital_type','release_label','survey_band','response_band','region','state']:
    counts=a.groupby(col,dropna=False).peer_sensitive_int.agg(['size','sum']).reset_index()
    for _,r in counts.iterrows():
        rows.append({'variable':col,'category':str(r[col]),'n':int(r['size']),
                     'events':int(r['sum']),'non_events':int(r['size']-r['sum']),
                     'one_class_only':int(r['sum']==0 or r['sum']==r['size'])})
pd.DataFrame(rows).to_csv(OUT/'tables'/'regression_sparse_cell_audit.csv',index=False)

# Reviewer source cell 13
all_models={}; coefficients=[]; summaries=[]; diagnostics=[]
def fit_cluster_glm(name,formula,frame):
    fitted=smf.glm(formula,data=frame,family=sm.families.Binomial(),eval_env=-1).fit(
        cov_type='cluster',cov_kwds={'groups':frame['facility']},maxiter=200,disp=0)
    ci=fitted.conf_int()
    beta=fitted.params.to_numpy(float); se=fitted.bse.to_numpy(float)
    condition=float(np.linalg.cond(fitted.model.exog))
    pred=fitted.predict(frame)
    flags=[]
    if not fitted.converged:flags.append('not_converged')
    if not (np.isfinite(beta).all() and np.isfinite(se).all()):flags.append('nonfinite_estimates')
    if np.max(np.abs(beta))>12:flags.append('large_coefficient')
    if np.max(se)>5:flags.append('large_cluster_se')
    if condition>1e10:flags.append('ill_conditioned_design')
    if np.mean((pred<1e-7)|(pred>1-1e-7))>0.01:flags.append('near_deterministic_predictions')
    term=[t for t in fitted.params.index if 'hospital_type' in t]
    if len(term)!=1:raise AssertionError(f'Cannot identify hospital type coefficient: {term}')
    t=term[0]
    for name_t in fitted.params.index:
        coefficients.append({'model':name,'term':name_t,'log_odds':float(fitted.params[name_t]),
             'cluster_se':float(fitted.bse[name_t]),'odds_ratio':float(np.exp(np.clip(fitted.params[name_t],-25,25))),
             'ci_low':float(np.exp(np.clip(ci.loc[name_t,0],-25,25))),
             'ci_high':float(np.exp(np.clip(ci.loc[name_t,1],-25,25))),
             'p_value':float(fitted.pvalues[name_t]),'diagnostic_flags':';'.join(flags) or 'none'})
    acute=frame.copy();critical=frame.copy()
    acute['hospital_type']='Acute Care Hospitals';critical['hospital_type']='Critical Access Hospitals'
    p_a=float(fitted.predict(acute).mean());p_c=float(fitted.predict(critical).mean())
    summaries.append({'model':name,'hospital_years':int(fitted.nobs),'unique_hospitals':int(frame.facility.nunique()),
        'critical_vs_acute_OR':float(np.exp(fitted.params[t])),
        'or_ci_low':float(np.exp(ci.loc[t,0])),'or_ci_high':float(np.exp(ci.loc[t,1])),
        'marginal_acute':p_a,'marginal_critical':p_c,
        'standardized_diff_critical_minus_acute_pp':100*(p_c-p_a),'converged':bool(fitted.converged),
        'max_abs_coefficient':float(np.abs(beta).max()),'max_cluster_se':float(se.max()),
        'condition_number':condition,'diagnostic_flags':';'.join(flags) or 'none'})
    diagnostics.append({'model':name,'formula':formula,'flags':flags,
        'near_deterministic_fraction':float(np.mean((pred<1e-7)|(pred>1-1e-7))),
        'max_abs_coefficient':float(np.abs(beta).max()),'max_cluster_se':float(se.max())})
    all_models[name]=fitted
for name,formula in FORMULAS.items():
    try:fit_cluster_glm(name,formula,a)
    except Exception as e:diagnostics.append({'model':name,'error':repr(e)});print('Fit failed:',name,repr(e))
numeric=a.loc[a.survey_band.ne('censored_or_unavailable')].copy()
try:fit_cluster_glm('D_numeric_count_only',FORMULAS['B_plus_survey_context'],numeric)
except Exception as e:diagnostics.append({'model':'D_numeric_count_only','error':repr(e)})
model_table=pd.DataFrame(summaries);coef_table=pd.DataFrame(coefficients)
model_table.to_csv(OUT/'tables'/'adjusted_hospital_type_models.csv',index=False)
coef_table.to_csv(OUT/'tables'/'adjusted_model_all_coefficients.csv',index=False)
(OUT/'qc'/'regression_diagnostics.json').write_text(json.dumps(diagnostics,indent=2))
if model_table.empty:raise RuntimeError('All adjusted models failed. Inspect regression_diagnostics.json.')
print(model_table[['model','critical_vs_acute_OR','or_ci_low','or_ci_high','standardized_diff_critical_minus_acute_pp','diagnostic_flags']].to_string(index=False))

## Reviewer validation — F. Reviewer correction — compare equally sized hospital-type peer sets / G. Hospital-level bootstrap of the observed unadjusted hospital-type contrast

In [ ]:
# Reviewer source cell 15
rng=np.random.default_rng(SEED+101)
peer_draw_rows=[]
for year,P in policies_by_year.items():
    md_year=meta.loc[meta.Year.eq(year)].drop_duplicates('facility').set_index('facility').reindex(P.index)
    acute=np.sort(md_year.index[md_year.hospital_type.eq('Acute Care Hospitals')].to_numpy(dtype=str))
    critical=np.sort(md_year.index[md_year.hospital_type.eq('Critical Access Hospitals')].to_numpy(dtype=str))
    if min(len(acute),len(critical))<=BALANCED_FIXED_N:
        raise ValueError(f'Insufficient balanced peer group in {year}: acute={len(acute)},critical={len(critical)}')
    critical_full_rate=float(sensitivity_for_policies(P.loc[critical])[0].mean())
    for rep in range(BALANCED_REPEATS):
        ac=np.sort(rng.choice(acute,len(critical),replace=False))
        ar=float(sensitivity_for_policies(P.loc[ac])[0].mean())
        peer_draw_rows.append({'year':year,'design':'acute_matched_to_all_critical','repeat':rep,
          'peer_n_acute':len(critical),'peer_n_critical':len(critical),
          'acute_sensitive':ar,'critical_sensitive':critical_full_rate,
          'difference_critical_minus_acute_pp':100*(critical_full_rate-ar)})
        ac2=np.sort(rng.choice(acute,BALANCED_FIXED_N,replace=False))
        cr2=np.sort(rng.choice(critical,BALANCED_FIXED_N,replace=False))
        rate_a=float(sensitivity_for_policies(P.loc[ac2])[0].mean())
        rate_c=float(sensitivity_for_policies(P.loc[cr2])[0].mean())
        peer_draw_rows.append({'year':year,'design':'both_random_fixed_600','repeat':rep,
          'peer_n_acute':BALANCED_FIXED_N,'peer_n_critical':BALANCED_FIXED_N,
          'acute_sensitive':rate_a,'critical_sensitive':rate_c,
          'difference_critical_minus_acute_pp':100*(rate_c-rate_a)})
draws=pd.DataFrame(peer_draw_rows)
draws.to_csv(OUT/'tables'/'equal_size_peer_draws.csv',index=False)
smry=[]
for (year,design),g in draws.groupby(['year','design']):
    vals=g.difference_critical_minus_acute_pp.to_numpy(float)
    smry.append({'year':year,'design':design,'draws':len(vals),'mean_difference_pp':float(np.mean(vals)),
      'q025_design_pp':float(np.quantile(vals,.025)),'q975_design_pp':float(np.quantile(vals,.975)),
      'positive_fraction':float(np.mean(vals>0)),'mean_acute_sensitive':float(g.acute_sensitive.mean()),
      'mean_critical_sensitive':float(g.critical_sensitive.mean())})
peer_sensitivity=pd.DataFrame(smry)
peer_sensitivity.to_csv(OUT/'tables'/'equal_size_peer_sensitivity_summary.csv',index=False)
print(peer_sensitivity.to_string(index=False))

# Reviewer source cell 17
hosp=a.groupby(['facility','hospital_type'],as_index=False).peer_sensitive_int.mean()
A=hosp.loc[hosp.hospital_type.eq('Acute Care Hospitals'),'peer_sensitive_int'].to_numpy(float)
B=hosp.loc[hosp.hospital_type.eq('Critical Access Hospitals'),'peer_sensitive_int'].to_numpy(float)
if min(len(A),len(B))==0:raise ValueError('Hospital-type group missing from bootstrap')
rng=np.random.default_rng(SEED+1001)
boot=np.array([A[rng.integers(len(A),size=len(A))].mean()-B[rng.integers(len(B),size=len(B))].mean()
              for _ in range(HOSPITAL_BOOTSTRAP)])
boot_result={'group_difference':'acute_minus_critical','unique_acute_hospitals':len(A),
    'unique_critical_hospitals':len(B),'observed_diff_pp':100*float(A.mean()-B.mean()),
    'ci_lower_pp':100*float(np.quantile(boot,.025)),'ci_upper_pp':100*float(np.quantile(boot,.975)),
    'draws':HOSPITAL_BOOTSTRAP,'cluster':'hospital','causal':False}
(OUT/'tables'/'hospital_cluster_bootstrap.json').write_text(json.dumps(boot_result,indent=2))
print(json.dumps(boot_result,indent=2))

## Reviewer validation — H. Measurement and policy admissibility audit / I. Cohort denominator reconciliation and a reproducible policy-flip case

In [ ]:
# Reviewer source cell 19
reviews=[]
for concept in locked:
    parts=concept.split(' || ')
    measure=parts[0]
    defs=concept_catalog.loc[concept_catalog.measure.eq(measure)]
    reviews.append({'measure_id':measure,'analysis_concept':concept,
      'question_text':parts[1] if len(parts)>1 else '',
      'answer_description':parts[2] if len(parts)>2 else '',
      'release_years':','.join(map(str,sorted(defs.Year.unique()))),
      'different_recorded_meanings':int(defs.sem_key.nunique()),
      'historical_official_construct_review':'PENDING','overlapping_construct_review':'PENDING',
      'review_conclusion':'lexically stable only'})
pd.DataFrame(reviews).to_csv(OUT/'tables'/'six_concept_semantic_checklist.csv',index=False)
policies=[]
for p in policy_registry.policy.unique():
    kind='fixed reference' if p=='fixed_mean' else 'catalog-expansion' if p=='all_available_mean' else 'single-concept-omission' if p.startswith('drop:') else 'aggregation-change'
    risk='indicator redundancy or shifted construct mix' if kind=='catalog-expansion' else 'meaningful dimension omitted' if kind=='single-concept-omission' else 'sensitivity to extremes' if kind=='aggregation-change' else 'investigator-chosen equal weights'
    policies.append({'policy':p,'purpose':kind,'methodological_risk':risk,
       'edit_cost':cost(p),'official_CMS_rating':'NO','admissible_for_computational_stress_test':'YES',
       'clinically_endorsed':'NOT ASSESSED','historical_CMS_guideline_review':'PENDING'})
pd.DataFrame(policies).to_csv(OUT/'tables'/'policy_review_checklist.csv',index=False)
audit={'fixed_concepts':len(locked),'year_policy_counts':policy_registry.groupby('year').policy.count().to_dict(),
  'historical_CMS_row_reconciliation':False,'clinical_construct_equivalence_independently_verified':False,
  'clinical_policy_approval':False,
  'CMS_archives':'https://data.cms.gov/provider-data/archived-data/hospitals',
  'CMS_HCAHPS':'https://www.cms.gov/data-research/research/consumer-assessment-healthcare-providers-systems/hospital-cahps-hcahps',
  'historical_data_described_as':'historical CMS-derived copies; no verified row-by-row reconciliation'}
(OUT/'qc'/'measurement_policy_audit.json').write_text(json.dumps(audit,indent=2))
print('Semantic audit reports produced; no unsupported claims of expert validation.')

# Reviewer source cell 21
recs=[]
for year,P in policies_by_year.items():
    eligible=meta.loc[meta.Year.eq(year)].drop_duplicates('facility')
    eligible=eligible.loc[eligible.facility.isin(P.index)]
    subset=a.loc[a.release_year.eq(year)]
    excluded=eligible.loc[~eligible.facility.isin(subset.facility)]
    count_missing=int(eligible.survey_n.isna().sum())
    recs.append({'year':year,'eligible_hospitals':len(eligible),'two_type_peer_hospitals':len(subset),
      'excluded_other_hospital_types':len(excluded),
      'excluded_types':' | '.join(sorted(excluded.hospital_type.dropna().astype(str).unique())),
      'missing_numeric_survey_count_in_eligible':count_missing,
      'peer_survey_band_count_sum':int(subset.survey_band.value_counts(dropna=False).sum())})
reconcile=pd.DataFrame(recs)
reconcile.to_csv(OUT/'tables'/'full_vs_peer_denominators.csv',index=False)
vol=a.groupby(['release_year','survey_band'],dropna=False).peer_sensitive_int.agg(['size','sum','mean']).reset_index()
vol.rename(columns={'size':'n_hospitals','sum':'sensitive_hospitals','mean':'sensitive_fraction'},inplace=True)
vol.to_csv(OUT/'tables'/'survey_volume_peer_sensitivity.csv',index=False)
print(reconcile.to_string(index=False))

case=flips.loc[(flips.year==2020)&(flips.facility=='100081')&(flips.k==100)]
if len(case)==1:
    case=case.iloc[0]
    P=policies_by_year[2020]
    def ordered_position(p,h):
        return int(P[p].sort_index().sort_values(ascending=False,kind='mergesort').index.get_loc(h)+1)
    r0=ordered_position('fixed_mean','100081');r1=ordered_position(case.flip_policy,'100081')
    assert (r0<=100)!=(r1<=100)
    case_record={'year':2020,'facility':'100081','k':100,'reference_rank':r0,
      'alternative_rank':r1,'alternative_policy':str(case.flip_policy),'edit_cost':int(case.minimum_registered_cost),
      'reference_measurements_unchanged':True}
    pd.DataFrame([case_record]).to_csv(OUT/'tables'/'verified_decision_reversal_example.csv',index=False)
    print(case_record)
else:
    (OUT/'notes'/'specific_case_unavailable.txt').write_text('The previously described example is not present in these input copies; use actual witness rows instead.')
    print('Specific manuscript case not present: no invented rank values.')

## Reviewer validation — J. Manuscript-ready figures and final independent evidence gates

In [ ]:
# Reviewer source cell 23
plt.rcParams.update({'font.size':9,'axes.spines.top':False,'axes.spines.right':False})
def save_fig(fig,name):
    fig.tight_layout()
    for ext in ('png','svg','pdf'):
        fig.savefig(OUT/'figures'/(name+'.'+ext),dpi=300,bbox_inches='tight')
    plt.close(fig)
if not model_table.empty:
    fig,ax=plt.subplots(figsize=(8,4.2))
    yy=np.arange(len(model_table))
    x=model_table.critical_vs_acute_OR.to_numpy(float)
    lo=model_table.or_ci_low.to_numpy(float);hi=model_table.or_ci_high.to_numpy(float)
    ax.errorbar(x,yy,xerr=[x-lo,hi-x],fmt='o',capsize=3)
    ax.axvline(1,color='gray',linestyle='--')
    ax.set_yticks(yy,model_table.model.tolist());ax.set_xscale('log')
    ax.set_xlabel('Critical access vs acute care: hospital-clustered odds ratio')
    ax.set_title('Hospital-type association across prespecified adjustments')
    save_fig(fig,'F1_adjusted_hospital_type_models')
fig,ax=plt.subplots(figsize=(8.5,4.3))
for label,g in peer_sensitivity.groupby('design'):
    gg=g.sort_values('year');line=ax.plot(gg.year,gg.mean_difference_pp,marker='o',label=label.replace('_',' '))[0]
    ax.fill_between(gg.year.to_numpy(float),gg.q025_design_pp.to_numpy(float),gg.q975_design_pp.to_numpy(float),
        alpha=.13,color=line.get_color())
ax.axhline(0,color='gray',linestyle='--',linewidth=1)
ax.set_xticks(EXPECTED_YEARS)
ax.set_xlabel('Reporting release');ax.set_ylabel('Critical access minus acute care (percentage points)')
ax.set_title('Equal-size peer-group re-ranking sensitivity')
ax.legend(fontsize=8);save_fig(fig,'F2_balanced_peer_size')
fig,ax=plt.subplots(figsize=(7.3,4))
heat=certsum.pivot(index='year',columns='k',values='robust_reference_topk_pct').reindex(index=EXPECTED_YEARS,columns=TOP_K)
im=ax.imshow(heat.to_numpy(),aspect='auto',vmin=0,vmax=100,cmap='viridis')
ax.set_xticks(np.arange(len(TOP_K)),TOP_K)
ax.set_yticks(np.arange(len(EXPECTED_YEARS)),EXPECTED_YEARS)
for i in range(len(EXPECTED_YEARS)):
    for j in range(len(TOP_K)):
        ax.text(j,i,f'{heat.iloc[i,j]:.0f}%',ha='center',va='center',color='white',fontsize=8)
fig.colorbar(im,ax=ax,label='Robust reference top-k (%)');ax.set_title('Finite-policy certificate stability')
save_fig(fig,'F3_certificate_heatmap')
if 'case_record' in globals():
    fig,ax=plt.subplots(figsize=(7.5,3.8))
    ax.plot([0,1],[r0,r1],marker='o',linewidth=2)
    ax.axhline(100,linestyle='--',color='gray',linewidth=1)
    ax.set_xticks([0,1],['Reference mean','Registered alternative'])
    ax.set_ylabel('Hospital rank (lower is better)');ax.invert_yaxis()
    ax.set_xlim(-.25,1.25)
    for xx,rr in [(0,r0),(1,r1)]:ax.annotate(f'Rank {rr}',(xx,rr),xytext=(10,-10),textcoords='offset points')
    ax.set_title('Verified hospital-level policy reversal (2020, top-100)')
    save_fig(fig,'F4_verified_policy_flip_example')

# Reviewer source cell 24
assert len(located)==5
assert len(cert)==sum(len(P)*len(TOP_K) for P in policies_by_year.values()), 'Unexpected certificate table length'
assert sum(z['membership_mismatches'] for z in independent)==0
assert sum(z['membership_mismatch'] for z in peer_check)==0
assert len(peer_sensitivity)==2*len(EXPECTED_YEARS)
assert (peer_sensitivity.draws==BALANCED_REPEATS).all()
assert len(flips)==len(cert.loc[cert.status.eq('sensitive')]), 'Every sensitive classification needs one flip witness'
assert not reconcile.empty
scope={'no_external_results_zip_required':True,'only_research_inputs':'five historical CMS-derived hospital CSV files',
 'run_seed':SEED,'source_rows':int(len(raw)),'eligible_hospital_years':len(panel),
 'peer_hospital_years':len(a),'certificate_comparisons':len(cert),
 'independent_topk_mismatches':0,'independent_peer_membership_mismatches':0,
 'verified_flip_witnesses':len(flips),'balanced_repeats':BALANCED_REPEATS,
 'reviewer_target':'Regressions, matched peer-group sizes, historical concept audit, policy review, denominators',
 'regression_model_flags':model_table[['model','diagnostic_flags']].to_dict(orient='records'),
 'clinical_ground_truth':False,'independent_holdout':False,'patient_level_standard_errors':False,
 'clinically_endorsed_policies':False,'clinical_implementation_evaluated':False,
 'acceptance_guarantee':False}
(OUT/'qc'/'FINAL_EVIDENCE_GATES.json').write_text(json.dumps(scope,indent=2))
lines=['# ChronoQuality-CRAC — Standalone reviewer-correction evidence',
 '','## Provenance','Only the five supplied historical hospital CSVs were read. No previous results ZIP was required.',
 '','## Exact certification',f"Certificate checks: {len(cert)}; mismatches: 0. Verified reversals: {len(flips)}.",
 '','## Subgroup model sensitivity',model_table.to_string(index=False),
 '','## Matched peer-group-size sensitivity',peer_sensitivity.to_string(index=False),
 '','## Reconciled hospital populations',reconcile.to_string(index=False),
 '','## Scientific boundaries','Model coefficients are observational, design bands are not patient-level CIs, and automated measure text matching does not establish expert-approved clinical comparability.']
(OUT/'notes'/'FINAL_REPORT.md').write_text('\n'.join(lines))
(OUT/'qc'/'output_inventory.json').write_text(json.dumps([str(f.relative_to(OUT)) for f in sorted(OUT.rglob('*')) if f.is_file()],indent=2))
FINAL=ROOT/'ChronoQuality_CRAC_Standalone_Reviewer_Validation_Results.zip'
with zipfile.ZipFile(FINAL,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=5,allowZip64=True) as z:
    for f in sorted(OUT.rglob('*')):
        if f.is_file():z.write(f,arcname=str(f.relative_to(OUT.parent)))
with zipfile.ZipFile(FINAL) as z:
    assert z.testzip() is None
print('Completed:',FINAL,'size_MB:',builtins.round(FINAL.stat().st_size/1048576,2),
      'certificate_checks:',len(cert),'policy_flips:',len(flips))

## Combined archive and reproducibility manifest

Preserves original V7 ZIP and reviewer ZIP and makes a convenient combined package.

In [ ]:
from pathlib import Path
import zipfile, json, hashlib
work=Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
v7_zip=work/'ChronoQuality_RUN_007_Results.zip'
# Reviewer notebook defines OUT and creates its ZIP in the preceding stage.
review_zip=work/'ChronoQuality_CRAC_Standalone_Reviewer_Validation_Results.zip'
if not review_zip.exists():
    candidates=sorted(work.rglob('*Reviewer*Results*.zip'))
    if len(candidates)==1: review_zip=candidates[0]
assert v7_zip.is_file(), f'Missing original V7 output: {v7_zip}'
assert review_zip.is_file(), f'Missing reviewer output: {review_zip}'
for fp in (v7_zip,review_zip):
    with zipfile.ZipFile(fp) as z:
        bad=z.testzip()
        assert bad is None, f'Corrupt archive entry: {fp}: {bad}'
combined=work/'ChronoQuality_V7_With_All_Reviewer_Validation.zip'
manifest={'base_pipeline':'original full V7 code, in order','reviewer_pipeline':'independent five-CSV reconstruction + focused reviewer validation',
          'inputs':'five historical hospital-year CSVs','output_files':[]}
with zipfile.ZipFile(combined,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=4) as z:
    for arc,filename in [(v7_zip,'original_V7/ChronoQuality_RUN_007_Results.zip'),(review_zip,'reviewer_validation/Reviewer_Validation_Results.zip')]:
        z.write(arc,filename)
        manifest['output_files'].append({'name':filename,'bytes':arc.stat().st_size,'sha256':hashlib.sha256(arc.read_bytes()).hexdigest()})
    z.writestr('combined_manifest.json',json.dumps(manifest,indent=2))
print('Combined evidence archive:',combined)
